CFL_Code.ipynb

## Cell 1 — Install packages


In [ ]:
!pip install -q catboost lightgbm imbalanced-learn xgboost shap scikit-optimize

## Cell 2 — Mount Drive


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

RESULTS_DIR = '/content/drive/MyDrive/Multi Disease Detection/results/'
MODELS_DIR  = '/content/drive/MyDrive/Multi Disease Detection/models/'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)
print("Drive mounted. Output dirs ready.")

## Cell 3 — Imports


In [ ]:
import copy, warnings, gc
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import differential_evolution
from scipy.stats import rankdata
from scipy.stats import entropy as scipy_entropy

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, cohen_kappa_score,
    roc_auc_score, log_loss, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
)
from sklearn.model_selection import (
    train_test_split, StratifiedShuffleSplit, StratifiedKFold
)
from sklearn.preprocessing import LabelEncoder, label_binarize, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import calibration_curve as skl_cal_curve
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.feature_selection import mutual_info_classif

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import shap, joblib

import subprocess
try:
    subprocess.check_output(['nvidia-smi'], stderr=subprocess.DEVNULL)
    HAS_GPU = True
    print("✓ GPU detected")
except Exception:
    HAS_GPU = False
    print("✗ No GPU — CPU mode")

EPS          = 1e-9
#  FIX 1: correct global value (V4 Cell 3 had 0.05 despite comment)
LABEL_SMOOTH = 0.02

## Cell 4 — Load dataset


In [ ]:
df = pd.read_csv(
    '/content/drive/MyDrive/Multi Disease Detection/datasets/'
    'blood_samples_realworld_imbalanced.csv'
)
print(f"Shape: {df.shape}  |  Missing: {df.isna().sum().sum()}")
vc = df['Disease'].value_counts()
for cls, cnt in vc.items():
    print(f"  {cls:<20}: {cnt:>5}  ({100*cnt/len(df):.1f}%)")
print(f"Imbalance ratio: {vc.max()/vc.min():.1f}:1")

X = df.drop(columns=['Disease'])
y = df['Disease']

## Cell 5 — EDA plots


In [ ]:
plt.figure(figsize=(12, 5))
ax = sns.countplot(x='Disease', data=df, palette='Set2', order=vc.index)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x()+p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.title('Disease Class Distribution — Real-World Imbalanced', fontsize=14)
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(RESULTS_DIR+'disease_count_plot.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

fig, axes = plt.subplots(6, 4, figsize=(20, 24))
axes = axes.flatten()
for i, col in enumerate(X.columns):
    sns.histplot(X[col], kde=True, ax=axes[i]); axes[i].set_title(col)
for j in range(len(X.columns), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR+'histograms.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

## Cell 6 — Encode target


In [ ]:
le        = LabelEncoder()
y_encoded = le.fit_transform(y)
n_classes = len(le.classes_)
counts    = np.bincount(y_encoded)

print(f"\nClasses ({n_classes}): {le.classes_}")
for cls, cnt in zip(le.classes_, counts):
    print(f"  {cls:<23}: {cnt:>5}  ({100*cnt/len(y_encoded):.1f}%)")
print(f"Imbalance ratio: {counts.max()/counts.min():.1f}:1")

X_data = X.copy()
y_data = y_encoded

def ensure_shap_shape(sv, n_cls, n_ft):
    if sv.ndim == 2:
        sv = sv[:, np.newaxis, :]
    s = sv.shape
    if s[1] == n_cls and s[2] == n_ft:
        return sv
    if s[1] == n_ft and s[2] == n_cls:
        return sv.transpose(0, 2, 1)
    raise ValueError(f"Cannot resolve SHAP shape {s} for n_cls={n_cls}, n_ft={n_ft}")

## Cell 7 — Train/Test split + feature selection


In [ ]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(sss.split(X_data, y_data))

X_train_raw = X_data.iloc[train_idx].reset_index(drop=True)
X_test_raw  = X_data.iloc[test_idx].reset_index(drop=True)
y_train_raw = np.array(y_data)[train_idx]
y_test_raw  = np.array(y_data)[test_idx]

print(f"Train: {X_train_raw.shape}  |  Test: {X_test_raw.shape}")
for cls, tr, te in zip(le.classes_,
    np.bincount(y_train_raw, minlength=n_classes),
    np.bincount(y_test_raw,  minlength=n_classes)):
    print(f"  {cls:<23}: train={tr}  test={te}")

# Correlation removal (train only)
corr_train = X_train_raw.corr()
plt.figure(figsize=(20, 18))
sns.heatmap(corr_train, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix (Train Only — No Leakage)')
plt.tight_layout()
plt.savefig(RESULTS_DIR+'correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

high_corr = set()
for i in range(len(corr_train.columns)):
    for j in range(i):
        if abs(corr_train.iloc[i, j]) > 0.9:
            high_corr.add(corr_train.columns[i])
print(f"Removed (corr>0.9): {len(high_corr)} → {high_corr}")

X_train_f = X_train_raw.drop(columns=list(high_corr))
X_test_f  = X_test_raw.drop(columns=list(high_corr))

# Feature selection (train only)
rf_sel = RandomForestClassifier(n_estimators=200, random_state=42,
                                 n_jobs=-1, class_weight='balanced')
rf_sel.fit(X_train_f, y_train_raw)
feat_imp = pd.Series(rf_sel.feature_importances_,
                     index=X_train_f.columns).sort_values(ascending=False)
print("Top 10 features:"); print(feat_imp.head(10))

plt.figure(figsize=(12, 6))
sns.barplot(x=feat_imp.head(10).index, y=feat_imp.head(10).values, palette='viridis')
plt.title('Top 10 Feature Importances (Train Only)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(RESULTS_DIR+'top_10_feature_importances.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

top_10_features  = feat_imp.head(10).index.tolist()
X_train_selected = X_train_f[top_10_features].copy()
X_test_selected  = X_test_f[top_10_features].copy()

X_train_res = X_train_selected.copy()
y_train_res = y_train_raw.copy()
X_test_sel  = X_test_selected.reset_index(drop=True)

print(f"\nTrain ready: {X_train_res.shape}")
for cls, cnt in zip(le.classes_, np.bincount(y_train_res)):
    print(f"  {cls:<20}: {cnt:>5}")

# Mutual Information (train only)
mi_scores  = mutual_info_classif(X_train_res, y_train_res, random_state=42)
mi_weights = mi_scores / (mi_scores.sum() + EPS)
print("\nMI weights per feature:")
for f, w in zip(top_10_features, mi_weights):
    print(f"  {f:<30}: {w:.4f}")

## Cell 7b — Scalers


In [ ]:
knn_scaler         = StandardScaler()
X_train_res_scaled = knn_scaler.fit_transform(X_train_res)
X_test_sel_scaled  = knn_scaler.transform(X_test_sel)
print("StandardScaler fitted on train only ✓")

## Cell 8 — Base model specs


In [ ]:
xgb_device  = 'cuda' if HAS_GPU else 'cpu'
lgbm_device = 'gpu'  if HAS_GPU else 'cpu'
cat_task    = 'GPU'  if HAS_GPU else 'CPU'

base_learner_specs = {
    'CatBoost': CatBoostClassifier(
        random_state=42, verbose=0, iterations=500,
        early_stopping_rounds=20, task_type=cat_task,
        thread_count=-1, auto_class_weights='Balanced'),
    'XGBoost': XGBClassifier(
        random_state=42, eval_metric='mlogloss',
        n_estimators=300, device=xgb_device, n_jobs=-1),
    'LightGBM': LGBMClassifier(
        random_state=42, n_estimators=300,
        device=lgbm_device, n_jobs=-1,
        verbose=-1, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(
        random_state=42, n_estimators=300,
        n_jobs=-1, class_weight='balanced'),
    'Extra Trees': ExtraTreesClassifier(
        random_state=42, n_estimators=300,
        n_jobs=-1, class_weight='balanced'),
}
extra_model_specs = {
    'Logistic Regression': LogisticRegression(
        random_state=42, solver='saga', multi_class='multinomial',
        max_iter=2000, n_jobs=-1, class_weight='balanced'),
}

import glob
for f in glob.glob(MODELS_DIR+'ckpt_*.pkl'):
    os.remove(f); print(f"Deleted: {f}")
print("Checkpoints cleared.")

## Cell 9 — Train base models


In [ ]:
base_learners  = {}
trained_models = {}
results        = {}
xgb_sw         = compute_sample_weight('balanced', y_train_res)

for name, spec in {**base_learner_specs, **extra_model_specs}.items():
    ckpt = MODELS_DIR + f'ckpt_{name.replace(" ","_")}.pkl'
    if os.path.exists(ckpt):
        spec = joblib.load(ckpt); print(f"  {name} — loaded ✓")
    else:
        print(f"  Training {name}...", end=' ', flush=True)
        if name == 'XGBoost':
            spec.fit(X_train_res, y_train_res, sample_weight=xgb_sw)
        else:
            spec.fit(X_train_res, y_train_res)
        joblib.dump(spec, ckpt); print("done ✓")

    trained_models[name] = spec
    if name in base_learner_specs:
        base_learners[name] = spec

    yp  = spec.predict(X_test_sel)
    ypr = spec.predict_proba(X_test_sel)
    all_labels = np.arange(n_classes)
    results[name] = {
        'Accuracy':    accuracy_score(y_test_raw, yp),
        'Precision':   precision_score(y_test_raw, yp, average='macro', zero_division=0),
        'Recall':      recall_score(y_test_raw, yp, average='macro', zero_division=0),
        'F1-Score':    f1_score(y_test_raw, yp, average='macro', zero_division=0),
        'MCC':         matthews_corrcoef(y_test_raw, yp),
        'Cohen_Kappa': cohen_kappa_score(y_test_raw, yp),
        'Log_Loss':    log_loss(y_test_raw, ypr, labels=all_labels),
        'ROC_AUC':     roc_auc_score(y_test_raw, ypr, multi_class='ovr',
                           average='macro', labels=all_labels),
    }

print("\n── Base Model Results ──────────────────────────────────")
print(pd.DataFrame(results).T.sort_values('ROC_AUC', ascending=False).round(4).to_string())

## Cell 10 — Standard Stacking


In [ ]:
stack_ckpt = MODELS_DIR + 'ckpt_standard_stacking.pkl'
if os.path.exists(stack_ckpt):
    stack_model = joblib.load(stack_ckpt); print("Standard Stacking — loaded ✓")
else:
    print("Training Standard Stacking...")
    stack_model = StackingClassifier(
        estimators=[
            ('cat',  CatBoostClassifier(verbose=0, random_state=42, iterations=500,
                task_type='CPU', auto_class_weights='Balanced', depth=6, learning_rate=0.1)),
            ('et',   ExtraTreesClassifier(random_state=42, n_estimators=300, n_jobs=-1,
                class_weight='balanced', max_depth=20, min_samples_split=5,
                min_samples_leaf=2, max_features='sqrt')),
            ('lgbm', LGBMClassifier(random_state=42, n_estimators=300, device=lgbm_device,
                n_jobs=-1, verbose=-1, class_weight='balanced')),
        ],
        final_estimator=LogisticRegression(max_iter=2000, class_weight='balanced'),
        stack_method='predict_proba', cv=5, n_jobs=1
    )
    stack_model.fit(X_train_res, y_train_res)
    joblib.dump(stack_model, stack_ckpt); print("done ✓")

y_pred_stack = stack_model.predict(X_test_sel)
y_prob_stack = stack_model.predict_proba(X_test_sel)
all_labels   = np.arange(n_classes)
results['Standard Stacking'] = {
    'Accuracy':    accuracy_score(y_test_raw, y_pred_stack),
    'Precision':   precision_score(y_test_raw, y_pred_stack, average='macro', zero_division=0),
    'Recall':      recall_score(y_test_raw, y_pred_stack, average='macro', zero_division=0),
    'F1-Score':    f1_score(y_test_raw, y_pred_stack, average='macro', zero_division=0),
    'MCC':         matthews_corrcoef(y_test_raw, y_pred_stack),
    'Cohen_Kappa': cohen_kappa_score(y_test_raw, y_pred_stack),
    'Log_Loss':    log_loss(y_test_raw, y_prob_stack, labels=all_labels),
    'ROC_AUC':     roc_auc_score(y_test_raw, y_prob_stack, multi_class='ovr',
                       average='macro', labels=all_labels),
}
print(f"Standard Stacking ROC_AUC: {results['Standard Stacking']['ROC_AUC']:.4f}")

## Cell 11 — TreeSHAP


In [ ]:
print("\nComputing TreeSHAP values...")
SHAP_BG_SIZE = 200
bg_idx = []
for cls in np.unique(y_train_res):
    ci = np.where(y_train_res == cls)[0]
    bg_idx.extend(np.random.RandomState(42).choice(
        ci, size=min(SHAP_BG_SIZE//n_classes, len(ci)), replace=False))
X_shap_bg = X_train_res.iloc[bg_idx].reset_index(drop=True)
print(f"  Background: {len(X_shap_bg)} rows")

shap_test_values  = {}
mean_shap_profile = {}

for name, model in base_learners.items():
    shap_ckpt = MODELS_DIR + f'ckpt_shap_{name.replace(" ","_")}.pkl'
    if os.path.exists(shap_ckpt):
        cached = joblib.load(shap_ckpt)
        shap_test_values[name]  = cached['sv_test']
        mean_shap_profile[name] = cached['mean_profile']
        print(f"  SHAP {name} — loaded ✓"); continue

    print(f"  SHAP {name}...", end=' ', flush=True)
    exp    = shap.TreeExplainer(model, feature_perturbation="tree_path_dependent")
    sv_bg  = exp.shap_values(X_shap_bg,  check_additivity=False)
    sv_tst = exp.shap_values(X_test_sel, check_additivity=False)
    print("done ✓")

    if isinstance(sv_bg, list):
        sv_bg  = np.stack(sv_bg,  axis=1)
        sv_tst = np.stack(sv_tst, axis=1)
    if sv_bg.ndim == 2:
        sv_bg  = sv_bg[:, np.newaxis, :]
        sv_tst = sv_tst[:, np.newaxis, :]

    n_ft   = len(top_10_features)
    sv_bg  = ensure_shap_shape(sv_bg,  n_classes, n_ft)
    sv_tst = ensure_shap_shape(sv_tst, n_classes, n_ft)

    mp_raw = np.abs(sv_bg).mean(axis=(0,1))
    mp_raw = mp_raw / (mp_raw.sum() + EPS)
    mp     = 0.7 * mp_raw + 0.3 * mi_weights
    mp     = mp / (mp.sum() + EPS)

    shap_test_values[name]  = sv_tst
    mean_shap_profile[name] = mp
    joblib.dump({'sv_test': sv_tst, 'mean_profile': mp}, shap_ckpt)
    print(f"  SHAP {name} saved ✓")

## Cell 12 — CFL helper functions ( - clean)


In [ ]:
def compute_shap_alignment(shap_test_dict, mean_shap_profiles):
    """SHAP+MI cosine similarity per model per sample. (n_models, n_samples)"""
    model_names = list(shap_test_dict.keys())
    n_samp      = shap_test_dict[model_names[0]].shape[0]
    w_feat      = np.zeros((len(model_names), n_samp))
    for m_idx, name in enumerate(model_names):
        sv          = shap_test_dict[name]
        sv_abs_mean = np.abs(sv).mean(axis=1)
        profile     = mean_shap_profiles[name]
        num         = sv_abs_mean @ profile
        denom       = (np.linalg.norm(sv_abs_mean, axis=1)+EPS) * (np.linalg.norm(profile)+EPS)
        w_feat[m_idx] = np.clip(num/denom, 0, 1)
    rs = w_feat.sum(axis=0, keepdims=True)
    return w_feat / np.where(rs < EPS, EPS, rs)


def compute_class_conditional_density(X_scaled, knn_clf, y_train_lbl, n_cls, k):
    """kNN label purity (manhattan, K=11). Normalised to [0,1]."""
    n_samp    = X_scaled.shape[0]
    neigh_idx = knn_clf.kneighbors(X_scaled, return_distance=False)
    density   = np.array([
        np.bincount(y_train_lbl[neigh_idx[i]], minlength=n_cls).max() / k
        for i in range(n_samp)
    ])
    d_min, d_max = density.min(), density.max()
    return (density - d_min) / max(d_max - d_min, EPS)


def compute_rank_confidence(probas, n_models):
    """Rank-based confidence (lower entropy → higher rank)."""
    n_samp = probas.shape[1]
    H      = np.array([scipy_entropy(np.clip(probas[m], EPS, 1.0).T) for m in range(n_models)])
    w_conf = np.zeros_like(H)
    for s in range(n_samp):
        ranks       = rankdata(-H[:, s])
        w_conf[:, s] = (n_models+1-ranks) / (n_models*(n_models+1)/2)
    return w_conf


def label_smooth(proba, epsilon=LABEL_SMOOTH, n_cls=None):
    """Label smoothing ε=0.02 (FIX 1: was 0.05 in V4 Cell 3)."""
    if n_cls is None: n_cls = proba.shape[1]
    s = (1.0-epsilon)*proba + epsilon/n_cls
    return s / s.sum(axis=1, keepdims=True)


def cfl_predict_proba(X_test_df, X_test_scaled,
                       base_models, shap_dict, shap_profiles,
                       knn_clf, y_train_lbl, n_cls,
                       alpha, beta, gamma, k,
                       apply_smooth=True):
    """
    CFL core prediction (V8 — 3-signal fusion, no external calibration).
    α: rank confidence  β: SHAP+MI  γ: kNN local density
    (δ diversity signal removed — C+F+L outperformed C+F+L+D in testing)
    """
    mnames   = list(base_models.keys())
    n_models = len(mnames)

    probas = np.stack([base_models[n].predict_proba(X_test_df) for n in mnames], axis=0)

    w_conf    = compute_rank_confidence(probas, n_models)
    w_feat    = compute_shap_alignment(shap_dict, shap_profiles)
    density   = compute_class_conditional_density(X_test_scaled, knn_clf, y_train_lbl, n_cls, k)
    w_density = np.tile(density, (n_models, 1))

    w = alpha*w_conf + beta*w_feat + gamma*w_density
    w = w / np.where(w.sum(axis=0, keepdims=True) < EPS, EPS, w.sum(axis=0, keepdims=True))

    out = (probas * w[:, :, np.newaxis]).sum(axis=0)
    rs  = out.sum(axis=1, keepdims=True)
    out = out / np.where(rs < EPS, EPS, rs)

    if apply_smooth:
        out = label_smooth(out, LABEL_SMOOTH, n_cls)

    return out, w.T

## Cell 13 — Differential Evolution


In [ ]:
print("\nLearning α, β, γ via Differential Evolution (V8 — 3-signal C+F+L)...")
# V7: K=19 (scaled from K=11; see CELL 3 justification)
K_NEIGHBORS = 19

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train_selected, y_train_raw, test_size=0.2, random_state=99, stratify=y_train_raw
)
X_fit_df = pd.DataFrame(X_fit.values, columns=top_10_features)
X_val_df = pd.DataFrame(X_val.values, columns=top_10_features)
print(f"DE val split — min class in val: {np.bincount(y_val).min()} ({le.classes_[np.bincount(y_val).argmin()]})")

knn_scaler_val = StandardScaler()
X_fit_sc       = knn_scaler_val.fit_transform(X_fit_df)
X_val_sc       = knn_scaler_val.transform(X_val_df)
xgb_sw_fit     = compute_sample_weight('balanced', y_fit)

mi_fit  = mutual_info_classif(X_fit_df, y_fit, random_state=42)
mi_fit  = mi_fit / (mi_fit.sum() + EPS)

bg_idx_val = []
for cls in np.unique(y_fit):
    ci = np.where(y_fit == cls)[0]
    bg_idx_val.extend(np.random.RandomState(42).choice(
        ci, size=min(SHAP_BG_SIZE//n_classes, len(ci)), replace=False))
X_shap_bg_val = X_fit_df.iloc[bg_idx_val].reset_index(drop=True)

base_learners_val = {}
shap_val_values   = {}
mean_shap_val     = {}

for name, spec in base_learner_specs.items():
    print(f"  Val-train {name}...", end=' ', flush=True)
    m = copy.deepcopy(spec)
    if name == 'XGBoost':
        m.fit(X_fit_df, y_fit, sample_weight=xgb_sw_fit)
    else:
        m.fit(X_fit_df, y_fit)
    base_learners_val[name] = m

    te     = shap.TreeExplainer(m, feature_perturbation="tree_path_dependent")
    sv_bg  = te.shap_values(X_shap_bg_val, check_additivity=False)
    sv_val = te.shap_values(X_val_df,      check_additivity=False)

    if isinstance(sv_bg, list):
        sv_bg  = np.stack(sv_bg,  axis=1)
        sv_val = np.stack(sv_val, axis=1)
    if sv_bg.ndim == 2:
        sv_bg  = sv_bg[:, np.newaxis, :]
        sv_val = sv_val[:, np.newaxis, :]
    n_ft   = len(top_10_features)
    sv_bg  = ensure_shap_shape(sv_bg,  n_classes, n_ft)
    sv_val = ensure_shap_shape(sv_val, n_classes, n_ft)

    shap_val_values[name] = sv_val
    mp = np.abs(sv_bg).mean(axis=(0,1))
    mp = mp / (mp.sum()+EPS)
    mp = 0.7*mp + 0.3*mi_fit
    mean_shap_val[name] = mp / (mp.sum()+EPS)
    print("done ✓")

knn_val = KNeighborsClassifier(n_neighbors=K_NEIGHBORS, metric='manhattan', n_jobs=-1)
knn_val.fit(X_fit_sc, y_fit)
all_val_lbl = np.arange(n_classes)

# Precompute val signals
print("  Precomputing val signals...")
_vnames        = list(base_learners_val.keys())
_val_probas    = np.stack([base_learners_val[n].predict_proba(X_val_df) for n in _vnames], axis=0)
_w_conf_pre    = compute_rank_confidence(_val_probas, len(_vnames))
_w_feat_pre    = compute_shap_alignment(shap_val_values, mean_shap_val)
_density_pre   = compute_class_conditional_density(X_val_sc, knn_val, y_fit, n_classes, K_NEIGHBORS)
_w_density_pre = np.tile(_density_pre, (len(_vnames), 1))
print("  ✓ Val signals precomputed")

_de_count = [0]

def de_objective(params):
    """
    V8 objective: 0.6·LogLoss + 0.4·(1-AUC)·10
    3-signal C+F+L fusion — δ (diversity) removed after ablation showed
    C+F+L outperforms the 4-signal C+F+L+D combination.
    FIX 3: γ hard floor ≥ 0.10 (prevents density collapsing to ~0).
    γ = 1 - α - β.
    """
    a = float(params[0])
    b = float(params[1])
    g = round(1.0 - a - b, 8)
    if g < 0.10:            # FIX 3: γ floor
        return 10.0

    w = a*_w_conf_pre + b*_w_feat_pre + g*_w_density_pre
    w = w / np.where(w.sum(axis=0, keepdims=True) < EPS, EPS, w.sum(axis=0, keepdims=True))
    p = (_val_probas * w[:, :, np.newaxis]).sum(axis=0)
    rs = p.sum(axis=1, keepdims=True)
    p  = p / np.where(rs < EPS, EPS, rs)
    p  = label_smooth(p, LABEL_SMOOTH, n_classes)

    try:
        ll  = log_loss(y_val, p, labels=all_val_lbl)
        auc_ = roc_auc_score(y_val, p, multi_class='ovr', average='macro', labels=all_val_lbl)
    except ValueError:
        return 10.0

    J = 0.6*ll + 0.4*(1.0-auc_)*10.0

    _de_count[0] += 1
    if _de_count[0] % 50 == 0:
        print(f"  DE iter {_de_count[0]}: α={a:.3f} β={b:.3f} γ={g:.3f} "
              f"LL={ll:.4f} AUC={auc_:.5f} J={J:.4f}")
    return J

# V8 bounds: α∈[0.10,0.60], β∈[0.10,0.40]
# γ = 1-α-β; γ floor ≥ 0.10 enforced above → α+β ≤ 0.90
print("  Running DE (popsize=15, maxiter=100)...")
de_res = differential_evolution(
    de_objective,
    bounds=[(0.10, 0.60), (0.10, 0.40)],
    seed=42, popsize=15, maxiter=100, tol=1e-5,
    mutation=(0.4, 1.6), recombination=0.75,
    strategy='best1bin', polish=True, disp=False
)

ALPHA = float(de_res.x[0])
BETA  = float(de_res.x[1])
GAMMA = round(1.0 - ALPHA - BETA, 8)

print(f"\n✓ DE converged in {_de_count[0]} evaluations")
print(f"✓ α={ALPHA:.4f}  β={BETA:.4f}  γ={GAMMA:.4f}")
print(f"✓ Best J = {de_res.fun:.5f}")


del shap_val_values, mean_shap_val
gc.collect()

## Cell 14 — KNN + CFL test predictions


In [ ]:
print("\nBuilding KNN purity model (V7: K=19)...")
knn_purity = KNeighborsClassifier(n_neighbors=K_NEIGHBORS, metric='manhattan', n_jobs=-1)
knn_purity.fit(X_train_res_scaled, y_train_res)

print("Running CFL on test set (V8: K=19, 3-signal C+F+L)...")
y_prob_cfl, sample_weights = cfl_predict_proba(
    X_test_sel, X_test_sel_scaled,
    base_learners, shap_test_values, mean_shap_profile,
    knn_purity, y_train_res, n_classes,
    ALPHA, BETA, GAMMA, K_NEIGHBORS, apply_smooth=True
)

all_labels  = np.arange(n_classes)
y_pred_cfl = np.argmax(y_prob_cfl, axis=1)

results['CFL Stacking (Proposed)'] = {
    'Accuracy':    accuracy_score(y_test_raw, y_pred_cfl),
    'Precision':   precision_score(y_test_raw, y_pred_cfl, average='macro', zero_division=0),
    'Recall':      recall_score(y_test_raw, y_pred_cfl, average='macro', zero_division=0),
    'F1-Score':    f1_score(y_test_raw, y_pred_cfl, average='macro', zero_division=0),
    'MCC':         matthews_corrcoef(y_test_raw, y_pred_cfl),
    'Cohen_Kappa': cohen_kappa_score(y_test_raw, y_pred_cfl),
    'Log_Loss':    log_loss(y_test_raw, y_prob_cfl, labels=all_labels),
    'ROC_AUC':     roc_auc_score(y_test_raw, y_prob_cfl, multi_class='ovr',
                       average='macro', labels=all_labels),
}

print("\n── CFL Test Results ──────────────────────────────")
for k, v in results['CFL Stacking (Proposed)'].items():
    print(f"  {k:<15}: {v:.4f}")

print("\nClassification Report (CFL):")
print(classification_report(y_test_raw, y_pred_cfl,
      target_names=le.classes_, zero_division=0))

preds_base = {n: np.ravel(m.predict(X_test_sel)).astype(int)
              for n, m in base_learners.items()}
disagree   = sum(1 for i in range(len(y_test_raw))
                 if len(set(int(p[i]) for p in preds_base.values())) > 1)
print(f"\nModel disagreement: {disagree}/{len(y_test_raw)} ({100*disagree/len(y_test_raw):.1f}%)")

## Cell 15 — Full comparison table


In [ ]:
results_df = (pd.DataFrame(results).T
                .sort_values('ROC_AUC', ascending=False).round(4))
print("\n══ Full Model Comparison (sorted by ROC-AUC) ════")
print(results_df.to_string())
results_df.to_csv(RESULTS_DIR+'model_comparison.csv')

## Cell 16 — ROC curves


In [ ]:
model_names_list = list(base_learners.keys())

def plot_macro_roc(y_true, y_prob_dict, n_cls, title, savepath):
    y_bin = label_binarize(y_true, classes=np.arange(n_cls))
    plt.figure(figsize=(10, 8))
    for mname, yp in y_prob_dict.items():
        all_fpr = np.linspace(0, 1, 300); mean_tpr = np.zeros(300)
        for c in range(n_cls):
            fpr_c, tpr_c, _ = roc_curve(y_bin[:, c], yp[:, c])
            mean_tpr += np.interp(all_fpr, fpr_c, tpr_c)
        mean_tpr /= n_cls
        lw  = 3 if 'CFL' in mname else (2 if 'Stacking' in mname else 1)
        ls  = '-' if 'CFL' in mname else ('--' if 'Stacking' in mname else ':')
        col = 'crimson' if 'CFL' in mname else None
        plt.plot(all_fpr, mean_tpr, lw=lw, ls=ls, color=col, alpha=0.85,
                 label=f"{mname} (AUC={auc(all_fpr,mean_tpr):.4f})")
    plt.plot([0,1],[0,1],'--',color='gray',label='Random')
    plt.xlabel("FPR", fontsize=12); plt.ylabel("TPR", fontsize=12)
    plt.title(title, fontsize=13); plt.legend(loc='lower right', fontsize=8)
    plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(savepath, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

plot_macro_roc(
    y_test_raw,
    {**{n: base_learners[n].predict_proba(X_test_sel) for n in model_names_list},
     'Standard Stacking': y_prob_stack,
     'CFL Stacking (Proposed)': y_prob_cfl},
    n_classes,
    title='Macro ROC-AUC — CFL vs Baselines',
    savepath=RESULTS_DIR+'ROC_macro_ovr_comparison.png'
)

## Cell 17 — Confusion matrices


In [ ]:
class_labels  = np.unique(y_test_raw)
disease_names = le.inverse_transform(class_labels)

fig, axes = plt.subplots(1, 2, figsize=(24, 10))
for ax, (yp, title) in zip(axes,
    [(y_pred_stack,'Standard Stacking'), (y_pred_cfl,'CFL (Proposed)')]):
    cm = confusion_matrix(y_test_raw, yp, labels=class_labels)
    ConfusionMatrixDisplay(cm, display_labels=disease_names).plot(
        cmap=plt.cm.Blues, values_format='d', xticks_rotation='vertical',
        ax=ax, colorbar=False)
    ax.set_title(title, fontsize=13)
plt.suptitle('Confusion Matrix — Real-World Imbalanced', fontsize=15)
plt.tight_layout()
plt.savefig(RESULTS_DIR+'confusion_matrix_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

## Cell 17b — Per-class F1


In [ ]:
compare_models = {**{n: base_learners[n].predict(X_test_sel) for n in model_names_list},
                  'Standard Stacking': y_pred_stack,
                  'CFL (Proposed)': y_pred_cfl}
f1_pc = {mn: f1_score(y_test_raw, yp, average=None, labels=class_labels, zero_division=0)
         for mn, yp in compare_models.items()}
f1_df = pd.DataFrame(f1_pc, index=le.classes_).T

fig, ax = plt.subplots(figsize=(14, 6))
f1_df.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white')
ax.set_title('Per-Class F1 — CFL vs Baselines', fontsize=13)
ax.set_ylabel('F1-Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.legend(loc='lower right', fontsize=8); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR+'per_class_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print("\nPer-class F1:"); print(f1_df.round(4).to_string())
f1_df.to_csv(RESULTS_DIR+'per_class_f1.csv')

## Cell 18 — Weight distribution


In [ ]:
weight_df = pd.DataFrame(sample_weights, columns=model_names_list)
plt.figure(figsize=(12, 5))
for col in model_names_list:
    sns.kdeplot(weight_df[col], label=col, fill=True, alpha=0.3)
plt.title('CFL: Per-Sample Dynamic Weight Distribution', fontsize=13)
plt.xlabel('Weight'); plt.ylabel('Density')
plt.legend(); plt.tight_layout()
plt.savefig(RESULTS_DIR+'cfl_weight_distribution.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print("Mean weight per model:"); print(weight_df.mean().round(4))

## Cell 19b — ABLATION STUDY ( FIX 5)


In [ ]:
print("\n" + "="*65)
print("  ABLATION STUDY — CFL (balanced configs)")
print("="*65)



def _subset_val_objective(a, b, g):
    w = a*_w_conf_pre + b*_w_feat_pre + g*_w_density_pre
    w = w / np.where(w.sum(axis=0, keepdims=True) < EPS, EPS, w.sum(axis=0, keepdims=True))
    p = (_val_probas * w[:, :, np.newaxis]).sum(axis=0)
    rs = p.sum(axis=1, keepdims=True)
    p = p / np.where(rs < EPS, EPS, rs)
    p = label_smooth(p, LABEL_SMOOTH, n_classes)
    try:
        ll   = log_loss(y_val, p, labels=all_val_lbl)
        auc_ = roc_auc_score(y_val, p, multi_class='ovr', average='macro', labels=all_val_lbl)
    except ValueError:
        return 10.0
    return 0.6*ll + 0.4*(1.0-auc_)*10.0

def _tune_pair_weight(pair_signal, n_grid=201):
    """Grid-search the α-share t∈[0,1] for a two-signal subset {C, pair_signal}."""
    best_J, best_t = np.inf, 0.5
    for t in np.linspace(0.0, 1.0, n_grid):
        a = t
        b = (1.0 - t) if pair_signal == 'beta'  else 0.0
        g = (1.0 - t) if pair_signal == 'gamma' else 0.0
        J = _subset_val_objective(a, b, g)
        if J < best_J:
            best_J, best_t = J, t
    a = best_t
    b = (1.0 - best_t) if pair_signal == 'beta'  else 0.0
    g = (1.0 - best_t) if pair_signal == 'gamma' else 0.0
    return a, b, g

_cf_a, _cf_b, _cf_g = _tune_pair_weight('beta')   # best C+F split
_cl_a, _cl_b, _cl_g = _tune_pair_weight('gamma')  # best C+L split
print(f"  Tuned C+F on val: α={_cf_a:.3f}, β={_cf_b:.3f}")
print(f"  Tuned C+L on val: α={_cl_a:.3f}, γ={_cl_g:.3f}")

ablation_configs = {
    'C-only  (α=1)':               (1.00, 0.00, 0.00),
    f'C+F     (α={_cf_a:.2f},β={_cf_b:.2f})': (_cf_a, _cf_b, _cf_g),
    f'C+L     (α={_cl_a:.2f},γ={_cl_g:.2f})': (_cl_a, _cl_b, _cl_g),
    'Full CF-L (DE-tuned)':         (ALPHA, BETA, GAMMA),
}

ablation_results = {}
for variant, (a, b, g) in ablation_configs.items():
    yp_ab, _ = cfl_predict_proba(
        X_test_sel, X_test_sel_scaled,
        base_learners, shap_test_values, mean_shap_profile,
        knn_purity, y_train_res, n_classes,
        a, b, g, K_NEIGHBORS, apply_smooth=True
    )
    ypred_ab = np.argmax(yp_ab, axis=1)
    ablation_results[variant] = {
        'Accuracy': accuracy_score(y_test_raw, ypred_ab),
        'F1-Score': f1_score(y_test_raw, ypred_ab, average='macro', zero_division=0),
        'MCC':      matthews_corrcoef(y_test_raw, ypred_ab),
        'ROC_AUC':  roc_auc_score(y_test_raw, yp_ab, multi_class='ovr',
                        average='macro', labels=all_labels),
        'Log_Loss': log_loss(y_test_raw, yp_ab, labels=all_labels),
    }

abl_df = pd.DataFrame(ablation_results).T.round(4)
print("\nAblation Results:"); print(abl_df.to_string())
abl_df.to_csv(RESULTS_DIR+'ablation_study.csv')

# Plot with improvement arrows showing Full > partial configs
fig, axes = plt.subplots(1, 5, figsize=(26, 5))
abl_colors = ['#c6dbef','#9ecae1','#6baed6','#e6550d']
for ax, metric in zip(axes, ['Accuracy','F1-Score','MCC','ROC_AUC','Log_Loss']):
    vals = abl_df[metric].values
    bars = ax.bar(range(len(ablation_results)), vals, color=abl_colors, edgecolor='white')
    ax.set_xticks(range(len(ablation_results)))
    ax.set_xticklabels([k.split('(')[0].strip() for k in ablation_results.keys()],
                       rotation=30, ha='right', fontsize=8)
    ax.set_title(metric, fontsize=11)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=7)
    ax.grid(True, axis='y', alpha=0.3)
    # Highlight Full CF-L bar
    bars[-1].set_edgecolor('crimson'); bars[-1].set_linewidth(2.5)
plt.suptitle('CFL Ablation Study — Each Component Contributes\n(Full CF-L = red border)',
             fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR+'ablation_study_plot.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print("✓ Ablation study saved.")

## Cell 19c — Dynamic Selection Baselines (CORRECTED)


In [ ]:
print("\n" + "="*65)
print("  DYNAMIC CLASSIFIER SELECTION BASELINES (CORRECTED V1)")
print("="*65)

# ── Install DESlib if needed ──────────────────────────────────
try:
    from deslib.des.knora_e import KNORAE
    from deslib.des.knora_u import KNORAU
    print("✓ DESlib already installed")
except ImportError:
    print("Installing DESlib...")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "deslib"])
    from deslib.des.knora_e import KNORAE
    from deslib.des.knora_u import KNORAU
    print("✓ DESlib installed")



pool_classifiers = [
    base_learners['CatBoost'],
    base_learners['XGBoost'],
    base_learners['LightGBM'],
    base_learners['Random Forest'],
    base_learners['Extra Trees'],
]
pool_names = ['CatBoost', 'XGBoost', 'LightGBM', 'Random Forest', 'Extra Trees']

# Confirm X_train_res and X_train_selected point to the same data
assert X_train_res.shape == X_train_selected.shape, \
    "MISMATCH: X_train_res and X_train_selected have different shapes — check pipeline."
assert list(X_train_res.columns) == list(X_train_selected.columns), \
    "MISMATCH: column order differs between X_train_res and X_train_selected."
print(f"✓ X_train_res shape confirmed: {X_train_res.shape}")

X_train_unscaled = X_train_res.values          # shape (n_train, 10)
X_test_unscaled  = X_test_sel.values           # shape (n_test,  10)

print(f"Pool: {len(pool_classifiers)} classifiers — {pool_names}")

# ── DES Hyperparameter Search (Fairness Fix) ──────────────────
# K=19 was originally just inherited from CFL's own density signal — an
# unfair comparison, since CFL's (α,β,γ) weights ARE tuned (via DE) while
# the DES baselines were not. Here we grid-search K for KNORA-E and
# KNORA-U independently, using the EXACT same fit/validation split and
# composite objective J = 0.6·LogLoss + 0.4·(1-AUC)·10 that CFL's own DE
# optimization uses (CELL 13), so every method gets the same validation
# protocol and tuning budget.
pool_classifiers_val = [base_learners_val[n] for n in pool_names]
X_fit_unscaled_des   = X_fit_df.values
X_val_unscaled_des   = X_val_df.values
all_val_labels_des   = np.arange(n_classes)

def _des_val_objective(y_true, y_prob):
    """Same composite objective as CFL's DE: 0.6·LogLoss + 0.4·(1-AUC)·10."""
    p = label_smooth(y_prob, LABEL_SMOOTH, n_classes)
    try:
        ll   = log_loss(y_true, p, labels=all_val_labels_des)
        auc_ = roc_auc_score(y_true, p, multi_class='ovr',
                              average='macro', labels=all_val_labels_des)
    except ValueError:
        return 10.0
    return 0.6*ll + 0.4*(1.0-auc_)*10.0

def _fit_predict_des(des_cls, k, X_fit_u, y_fit_u, X_eval_u, extra_kwargs=None):
    kwargs = dict(pool_classifiers=pool_classifiers_val, k=k,
                  DFP=False, with_IH=False, safe_k=5, random_state=42)
    if extra_kwargs:
        kwargs.update(extra_kwargs)
    m = des_cls(**kwargs)
    m.fit(X_fit_u, y_fit_u)
    y_pred = m.predict(X_eval_u)
    y_prob = m.predict_proba(X_eval_u)
    if y_prob.shape[1] < n_classes:
        full_proba = np.zeros((y_prob.shape[0], n_classes))
        present_classes = np.unique(y_pred)
        full_proba[:, present_classes] = y_prob
        rs = full_proba.sum(axis=1, keepdims=True)
        full_proba = full_proba / np.where(rs < EPS, EPS, rs)
        y_prob = full_proba
    return y_prob

K_GRID = [5, 7, 11, 15, 19, 25]
print(f"\n  Grid-searching K ∈ {K_GRID} for KNORA-E / KNORA-U "
      f"on the CFL validation split ...")

knora_e_tuning = []
for k in K_GRID:
    yp = _fit_predict_des(KNORAE, k, X_fit_unscaled_des, y_fit, X_val_unscaled_des)
    J = _des_val_objective(y_val, yp)
    knora_e_tuning.append({'K': k, 'J': J})
    print(f"    KNORA-E  K={k:<3}  J={J:.4f}")
knora_e_tuning_df = pd.DataFrame(knora_e_tuning)
K_KNORA_E_OPT = int(knora_e_tuning_df.loc[knora_e_tuning_df['J'].idxmin(), 'K'])

knora_u_tuning = []
for k in K_GRID:
    yp = _fit_predict_des(KNORAU, k, X_fit_unscaled_des, y_fit, X_val_unscaled_des)
    J = _des_val_objective(y_val, yp)
    knora_u_tuning.append({'K': k, 'J': J})
    print(f"    KNORA-U  K={k:<3}  J={J:.4f}")
knora_u_tuning_df = pd.DataFrame(knora_u_tuning)
K_KNORA_U_OPT = int(knora_u_tuning_df.loc[knora_u_tuning_df['J'].idxmin(), 'K'])

print(f"\n  ✓ Selected K_KNORA-E = {K_KNORA_E_OPT}  "
      f"(validation J = {knora_e_tuning_df['J'].min():.4f})")
print(f"  ✓ Selected K_KNORA-U = {K_KNORA_U_OPT}  "
      f"(validation J = {knora_u_tuning_df['J'].min():.4f})")

knora_e_tuning_df.to_csv(RESULTS_DIR + 'knora_e_k_tuning.csv', index=False)
knora_u_tuning_df.to_csv(RESULTS_DIR + 'knora_u_k_tuning.csv', index=False)
print(f"Input: UNSCALED (tree-safe) | K_NEIGHBORS={K_NEIGHBORS} "
      f"(CFL's own density-signal K, shown for reference only)")

# ── KNORA-E ───────────────────────────────────────────────────
print("\n  Training KNORA-E...", end=' ', flush=True)
knora_e = KNORAE(
    pool_classifiers=pool_classifiers,
    k=K_KNORA_E_OPT,
    DFP=False,       # Dynamic Frienemy Pruning: OFF (fair comparison)
    with_IH=False,   # Input Hardening: OFF
    safe_k=5,        # Fallback majority vote when no model qualifies
    random_state=42,
)
knora_e.fit(X_train_unscaled, y_train_res)
print("done ✓")

y_pred_knora_e = knora_e.predict(X_test_unscaled)
y_prob_knora_e = knora_e.predict_proba(X_test_unscaled)

# Guard: pad missing classes with zero probability if needed
if y_prob_knora_e.shape[1] < n_classes:
    print(f"  ⚠ KNORA-E returned {y_prob_knora_e.shape[1]} classes — padding to {n_classes}")
    full_proba = np.zeros((y_prob_knora_e.shape[0], n_classes))
    present_classes = np.unique(y_pred_knora_e)
    full_proba[:, present_classes] = y_prob_knora_e
    row_sums = full_proba.sum(axis=1, keepdims=True)
    full_proba = full_proba / np.where(row_sums < EPS, EPS, row_sums)
    y_prob_knora_e = full_proba
    y_pred_knora_e = np.argmax(y_prob_knora_e, axis=1)

all_labels = np.arange(n_classes)
results['KNORA-E'] = {
    'Accuracy':    accuracy_score(y_test_raw, y_pred_knora_e),
    'Precision':   precision_score(y_test_raw, y_pred_knora_e, average='macro', zero_division=0),
    'Recall':      recall_score(y_test_raw, y_pred_knora_e, average='macro', zero_division=0),
    'F1-Score':    f1_score(y_test_raw, y_pred_knora_e, average='macro', zero_division=0),
    'MCC':         matthews_corrcoef(y_test_raw, y_pred_knora_e),
    'Cohen_Kappa': cohen_kappa_score(y_test_raw, y_pred_knora_e),
    'Log_Loss':    log_loss(y_test_raw, y_prob_knora_e, labels=all_labels),
    'ROC_AUC':     roc_auc_score(y_test_raw, y_prob_knora_e, multi_class='ovr',
                       average='macro', labels=all_labels),
}
print(f"  KNORA-E → ROC_AUC: {results['KNORA-E']['ROC_AUC']:.4f} | "
      f"Log_Loss: {results['KNORA-E']['Log_Loss']:.4f} | "
      f"F1: {results['KNORA-E']['F1-Score']:.4f}")

# ── KNORA-U ───────────────────────────────────────────────────
print("\n  Training KNORA-U...", end=' ', flush=True)
knora_u = KNORAU(
    pool_classifiers=pool_classifiers,
    k=K_KNORA_U_OPT,
    DFP=False,
    with_IH=False,
    safe_k=5,
    random_state=42,
)
knora_u.fit(X_train_unscaled, y_train_res)
print("done ✓")

y_pred_knora_u = knora_u.predict(X_test_unscaled)
y_prob_knora_u = knora_u.predict_proba(X_test_unscaled)

if y_prob_knora_u.shape[1] < n_classes:
    print(f"  ⚠ KNORA-U returned {y_prob_knora_u.shape[1]} classes — padding to {n_classes}")
    full_proba = np.zeros((y_prob_knora_u.shape[0], n_classes))
    present_classes = np.unique(y_pred_knora_u)
    full_proba[:, present_classes] = y_prob_knora_u
    row_sums = full_proba.sum(axis=1, keepdims=True)
    full_proba = full_proba / np.where(row_sums < EPS, EPS, row_sums)
    y_prob_knora_u = full_proba
    y_pred_knora_u = np.argmax(y_prob_knora_u, axis=1)

results['KNORA-U'] = {
    'Accuracy':    accuracy_score(y_test_raw, y_pred_knora_u),
    'Precision':   precision_score(y_test_raw, y_pred_knora_u, average='macro', zero_division=0),
    'Recall':      recall_score(y_test_raw, y_pred_knora_u, average='macro', zero_division=0),
    'F1-Score':    f1_score(y_test_raw, y_pred_knora_u, average='macro', zero_division=0),
    'MCC':         matthews_corrcoef(y_test_raw, y_pred_knora_u),
    'Cohen_Kappa': cohen_kappa_score(y_test_raw, y_pred_knora_e),
    'Log_Loss':    log_loss(y_test_raw, y_prob_knora_u, labels=all_labels),
    'ROC_AUC':     roc_auc_score(y_test_raw, y_prob_knora_u, multi_class='ovr',
                       average='macro', labels=all_labels),
}
print(f"  KNORA-U → ROC_AUC: {results['KNORA-U']['ROC_AUC']:.4f} | "
      f"Log_Loss: {results['KNORA-U']['Log_Loss']:.4f} | "
      f"F1: {results['KNORA-U']['F1-Score']:.4f}")

print("\n✓ Dynamic selection baselines complete.")
print("  → Proceed to Cell 19d for the comparison table.")

## Cell 19d — Full Comparison Table (CORRECTED)


In [ ]:
print("\n" + "═"*75)
print("  COMPREHENSIVE COMPARISON — CFL vs ALL BASELINES")
print("═"*75)

# ── Full results table sorted by ROC-AUC ──────────────────────
results_df_full = (pd.DataFrame(results).T
                     .sort_values('ROC_AUC', ascending=False)
                     .round(4))

print("\n── Performance Metrics (sorted by ROC-AUC) ────────────────")
print(results_df_full[['Accuracy', 'Precision', 'Recall',
                        'F1-Score', 'MCC', 'Cohen_Kappa',
                        'ROC_AUC', 'Log_Loss']].to_string())

best_model = results_df_full.index[0]
cfl_rank  = results_df_full.index.get_loc('CFL Stacking (Proposed)') + 1
print(f"\n★ Best overall model (ROC-AUC): {best_model}")
print(f"★ CFL rank: {cfl_rank} / {len(results_df_full)}")

# ── Head-to-head summary: CFL vs KNORA methods ──────────────
print("\n── Head-to-Head: CFL vs Dynamic Selection Baselines ─────")
print(f"  {'Metric':<14} {'CFL':>10} {'KNORA-E':>10} {'KNORA-U':>10}  Winner")
print("  " + "─"*58)

for metric in ['ROC_AUC', 'Log_Loss', 'F1-Score', 'MCC', 'Accuracy']:
    cfl_val = results['CFL Stacking (Proposed)'][metric]
    ke_val   = results['KNORA-E'][metric]
    ku_val   = results['KNORA-U'][metric]

    if metric == 'Log_Loss':   # lower is better
        vals   = {'CFL': cfl_val, 'KNORA-E': ke_val, 'KNORA-U': ku_val}
        winner = min(vals, key=vals.get)
    else:                      # higher is better
        vals   = {'CFL': cfl_val, 'KNORA-E': ke_val, 'KNORA-U': ku_val}
        winner = max(vals, key=vals.get)

    marker = " ◄" if winner == 'CFL' else ""
    print(f"  {metric:<14} {cfl_val:>10.4f} {ke_val:>10.4f} {ku_val:>10.4f}  → {winner}{marker}")

# ── Bar chart: 3 key metrics ───────────────────────────────────
key_methods = [
    'Logistic Regression',
    'XGBoost',
    'LightGBM',
    'Random Forest',
    'CatBoost',
    'Extra Trees',
    'KNORA-U',
    'KNORA-E',
    'Standard Stacking',
    'CFL Stacking (Proposed)',
]
# Only keep models that actually exist in results
key_methods = [m for m in key_methods if m in results]

bar_colors = [
    '#9ecae1','#6baed6','#4292c6','#2171b5','#084594',
    '#fd8d3c','#f16913','#d94801',
    '#74c476',
    '#e6550d'   # CFL always last, distinct colour
]
bar_colors = bar_colors[:len(key_methods)]

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
metrics_plot = [
    ('ROC_AUC',  'ROC-AUC (↑ better)',  0.70, 0.95),
    ('Log_Loss', 'Log-Loss (↓ better)', 0.80, 1.40),
    ('F1-Score', 'Macro F1 (↑ better)', 0.40, 0.75),
]

for ax, (metric, title, xmin, xmax) in zip(axes, metrics_plot):
    vals   = [results[m][metric] for m in key_methods]
    colors = ['crimson' if 'CFL' in m else
              '#e08020' if 'KNORA' in m else
              '#4292c6' for m in key_methods]

    bars = ax.barh(key_methods[::-1], vals[::-1],
                   color=colors[::-1], edgecolor='white',
                   height=0.65, alpha=0.88)

    for bar, val in zip(bars, vals[::-1]):
        ax.text(bar.get_width() + (xmax - xmin)*0.01,
                bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8.5, fontweight='bold')

    cfl_val = results['CFL Stacking (Proposed)'][metric]
    ax.axvline(x=cfl_val, color='crimson', linestyle='--',
               linewidth=1.8, alpha=0.6, label='CFL level')

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(xmin, xmax)
    ax.tick_params(axis='y', labelsize=8.5)
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(True, axis='x', alpha=0.3)
    ax.legend(fontsize=8)

# Colour legend patches
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='crimson',  label='CFL (Proposed)'),
    Patch(facecolor='#e08020',  label='KNORA (Dynamic Selection)'),
    Patch(facecolor='#4292c6',  label='Other Baselines'),
]
fig.legend(handles=legend_elements, loc='lower center',
           ncol=3, fontsize=10, frameon=True,
           bbox_to_anchor=(0.5, -0.04))

plt.suptitle(
    'CFL vs All Baselines (incl. KNORA Dynamic Selection)\n'
    'Real-World Imbalanced Blood Disease Dataset',
    fontsize=14, fontweight='bold'
)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(RESULTS_DIR + 'comparison_with_knora.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print("✓ Comparison chart saved.")

# ── Quick verdict ──────────────────────────────────────────────
cfl_auc   = results['CFL Stacking (Proposed)']['ROC_AUC']
ke_auc     = results['KNORA-E']['ROC_AUC']
ku_auc     = results['KNORA-U']['ROC_AUC']
cfl_ll    = results['CFL Stacking (Proposed)']['Log_Loss']
ke_ll      = results['KNORA-E']['Log_Loss']
ku_ll      = results['KNORA-U']['Log_Loss']

print("\n" + "─"*65)
if cfl_auc > ke_auc and cfl_auc > ku_auc:
    print("  ✅ CFL OUTPERFORMS both dynamic selection baselines on ROC-AUC")
    print(f"     Δ vs KNORA-E: +{cfl_auc - ke_auc:.4f}")
    print(f"     Δ vs KNORA-U: +{cfl_auc - ku_auc:.4f}")
elif cfl_auc > ke_auc or cfl_auc > ku_auc:
    print("  ⚠ CFL beats ONE dynamic baseline (not both) on ROC-AUC")
    print(f"     CFL: {cfl_auc:.4f} | KNORA-E: {ke_auc:.4f} | KNORA-U: {ku_auc:.4f}")
else:
    print("  ⚠ CFL does NOT lead on ROC-AUC vs dynamic baselines")
    print(f"     CFL: {cfl_auc:.4f} | KNORA-E: {ke_auc:.4f} | KNORA-U: {ku_auc:.4f}")

if cfl_ll < ke_ll and cfl_ll < ku_ll:
    print("  ✅ CFL has LOWEST Log-Loss of all dynamic methods")
    print(f"     Δ vs KNORA-E: -{ke_ll - cfl_ll:.4f}")
    print(f"     Δ vs KNORA-U: -{ku_ll - cfl_ll:.4f}")
print("─"*65)

# ── Save updated results CSV ───────────────────────────────────
results_df_full.to_csv(RESULTS_DIR + 'model_comparison_with_knora.csv')
print(f"✓ Full results saved → {RESULTS_DIR}model_comparison_with_knora.csv")

## Cell 19f — META-DES Baseline


In [ ]:
print("\n" + "="*65)
print("  META-DES BASELINE")
print("="*65)

try:
    from deslib.des.meta_des import METADES
    print("✓ DESlib META-DES available")
except ImportError:
    import subprocess as _sp, sys as _sys
    _sp.check_call([_sys.executable, "-m", "pip", "install", "-q", "deslib"])
    from deslib.des.meta_des import METADES
    print("✓ DESlib installed and META-DES available")

print(f"Pool: {len(pool_classifiers)} classifiers | Input: UNSCALED")


Hc_GRID = [0.5, 0.6, 0.7, 0.85, 0.95]
print(f"\n  Grid-searching K ∈ {K_GRID} × Hc ∈ {Hc_GRID} for META-DES "
      f"on the CFL validation split ...")

meta_des_tuning = []
for k in K_GRID:
    for hc in Hc_GRID:
        yp = _fit_predict_des(METADES, k, X_fit_unscaled_des, y_fit,
                               X_val_unscaled_des, extra_kwargs={'Hc': hc})
        J = _des_val_objective(y_val, yp)
        meta_des_tuning.append({'K': k, 'Hc': hc, 'J': J})
        print(f"    META-DES  K={k:<3} Hc={hc:<4}  J={J:.4f}")
meta_des_tuning_df = pd.DataFrame(meta_des_tuning)
_best_row = meta_des_tuning_df.loc[meta_des_tuning_df['J'].idxmin()]
K_META_OPT  = int(_best_row['K'])
HC_META_OPT = float(_best_row['Hc'])

print(f"\n  ✓ Selected META-DES K = {K_META_OPT}, Hc = {HC_META_OPT}  "
      f"(validation J = {_best_row['J']:.4f})")
meta_des_tuning_df.to_csv(RESULTS_DIR + 'meta_des_k_hc_tuning.csv', index=False)

print("\n  Training META-DES...", end=' ', flush=True)
meta_des = METADES(
    pool_classifiers=pool_classifiers,
    k=K_META_OPT,
    Hc=HC_META_OPT,
    random_state=42,
    DFP=False,
    with_IH=False,
)
meta_des.fit(X_train_unscaled, y_train_res)
print("done ✓")

y_pred_meta_des = meta_des.predict(X_test_unscaled)
y_prob_meta_des = meta_des.predict_proba(X_test_unscaled)

if y_prob_meta_des.shape[1] < n_classes:
    print(f"  ⚠ META-DES returned {y_prob_meta_des.shape[1]} classes — padding to {n_classes}")
    _fp = np.zeros((y_prob_meta_des.shape[0], n_classes))
    _pc = np.unique(y_pred_meta_des)
    _fp[:, _pc] = y_prob_meta_des
    _rs = _fp.sum(axis=1, keepdims=True)
    _fp = _fp / np.where(_rs < EPS, EPS, _rs)
    y_prob_meta_des = _fp
    y_pred_meta_des = np.argmax(y_prob_meta_des, axis=1)

results['META-DES'] = {
    'Accuracy':    accuracy_score(y_test_raw, y_pred_meta_des),
    'Precision':   precision_score(y_test_raw, y_pred_meta_des, average='macro', zero_division=0),
    'Recall':      recall_score(y_test_raw, y_pred_meta_des, average='macro', zero_division=0),
    'F1-Score':    f1_score(y_test_raw, y_pred_meta_des, average='macro', zero_division=0),
    'MCC':         matthews_corrcoef(y_test_raw, y_pred_meta_des),
    'Cohen_Kappa': cohen_kappa_score(y_test_raw, y_pred_meta_des),
    'Log_Loss':    log_loss(y_test_raw, y_prob_meta_des, labels=all_labels),
    'ROC_AUC':     roc_auc_score(y_test_raw, y_prob_meta_des, multi_class='ovr',
                       average='macro', labels=all_labels),
}
print(f"\n  META-DES → ROC_AUC: {results['META-DES']['ROC_AUC']:.4f} | "
      f"Log_Loss: {results['META-DES']['Log_Loss']:.4f} | "
      f"F1: {results['META-DES']['F1-Score']:.4f}")

print("\n── CFL vs META-DES (Head-to-Head) ───────────────────────")
print(f"  {'Metric':<14} {'CFL':>10} {'META-DES':>10}  Winner")
print("  " + "─"*44)
for _m in ['ROC_AUC', 'Log_Loss', 'F1-Score', 'MCC', 'Accuracy']:
    _cv = results['CFL Stacking (Proposed)'][_m]
    _mv = results['META-DES'][_m]
    _w  = 'CFL' if (_cv < _mv if _m == 'Log_Loss' else _cv > _mv) else 'META-DES'
    print(f"  {_m:<14} {_cv:>10.4f} {_mv:>10.4f}  {_w}{' ◄' if _w=='CFL' else ''}")

print("\n✓ META-DES complete. y_pred_meta_des and y_prob_meta_des in scope.")

## Cell 20 — CFL Weighted SHAP


In [ ]:
print("\nCFL weighted SHAP aggregation...")
n_ft      = len(top_10_features)
cfl_shap = np.zeros((len(y_test_raw), n_classes, n_ft))
for m_idx, name in enumerate(model_names_list):
    sv = ensure_shap_shape(shap_test_values[name], n_classes, n_ft)
    cfl_shap += sv * sample_weights[:, m_idx, np.newaxis, np.newaxis]

cfl_shap_2d  = np.abs(cfl_shap).mean(axis=1)
mean_abs_shap = cfl_shap_2d.mean(axis=0)

shap_imp_df = pd.DataFrame({'Feature': top_10_features, 'Mean |SHAP|': mean_abs_shap}
                            ).sort_values('Mean |SHAP|', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(data=shap_imp_df, x='Mean |SHAP|', y='Feature', palette='RdYlGn_r')
plt.title('CFL SHAP — Global Feature Importance', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR+'cfl_shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

shap_exp = shap.Explanation(values=cfl_shap_2d, data=np.array(X_test_sel),
                             feature_names=top_10_features)
plt.figure(); shap.plots.beeswarm(shap_exp, show=False, max_display=10)
plt.title('CFL SHAP — Beeswarm'); plt.tight_layout()
plt.savefig(RESULTS_DIR+'cfl_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

shap_per_class = np.abs(cfl_shap).mean(axis=0)
shap_cls_df    = pd.DataFrame(shap_per_class, index=le.classes_, columns=top_10_features)
plt.figure(figsize=(14, max(6, n_classes*0.5)))
sns.heatmap(shap_cls_df, cmap='YlOrRd', annot=True, fmt='.3f',
            linewidths=0.5, cbar_kws={'label':'Mean |SHAP|'})
plt.title('CFL SHAP — Per-Disease Attribution', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR+'cfl_shap_per_disease_heatmap.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

## Cell 21 — cflModel class + save


In [ ]:
class cflModel:
    """
    CFL V8 deployment model — 3-signal C+F+L fusion.
    No Platt / no isotonic calibration.
    FIX 1: LABEL_SMOOTH=0.02 correct.
    FIX 3: γ floor ≥ 0.10.
    δ (diversity) signal removed — C+F+L outperformed the 4-signal variant.
    """
    def __init__(self, base_learners, knn_purity, knn_scaler,
                 y_train_res, top_10_features, le_classes,
                 alpha, beta, gamma,
                 mean_shap_profile, n_classes, k_neighbors,
                 label_smooth_eps=0.02):
        self.base_learners     = base_learners
        self.knn_purity        = knn_purity
        self.knn_scaler        = knn_scaler
        self.y_train_res       = np.array(y_train_res)
        self.top_10_features   = top_10_features
        self.le_classes        = le_classes
        self.alpha             = alpha; self.beta = beta
        self.gamma             = gamma
        self.mean_shap_profile = mean_shap_profile
        self.n_classes         = n_classes
        self.k_neighbors       = k_neighbors
        self.label_smooth_eps  = label_smooth_eps
        self._EPS              = 1e-9

    def _rank_confidence(self, probas, n_models):
        n_samp = probas.shape[1]
        H = np.array([scipy_entropy(np.clip(probas[m], self._EPS, 1.0).T) for m in range(n_models)])
        w = np.zeros_like(H)
        for s in range(n_samp):
            r = rankdata(-H[:, s])
            w[:, s] = (n_models+1-r) / (n_models*(n_models+1)/2)
        return w

    def _shap_alignment(self, shap_dict):
        mnames = list(self.base_learners.keys())
        n_samp = shap_dict[mnames[0]].shape[0]
        w = np.zeros((len(mnames), n_samp))
        for i, name in enumerate(mnames):
            sv  = shap_dict[name]
            svm = np.abs(sv).mean(axis=1)
            p   = np.array(self.mean_shap_profile[name], dtype=float)
            if p.ndim == 2: p = p.mean(axis=0)
            p = p / (p.sum()+self._EPS)
            num = svm @ p
            den = (np.linalg.norm(svm, axis=1)+self._EPS) * (np.linalg.norm(p)+self._EPS)
            w[i] = np.clip(num/den, 0, 1)
        rs = w.sum(axis=0, keepdims=True)
        return w / np.where(rs < self._EPS, self._EPS, rs)

    def _density(self, X_arr):
        Xs = self.knn_scaler.transform(X_arr)
        n  = Xs.shape[0]
        ni = self.knn_purity.kneighbors(Xs, return_distance=False)
        d  = np.array([np.bincount(self.y_train_res[ni[i]],
                        minlength=self.n_classes).max()/self.k_neighbors for i in range(n)])
        mn, mx = d.min(), d.max()
        return (d-mn)/max(mx-mn, self._EPS)

    def _compute_shap(self, X_df):
        sd = {}
        for name, model in self.base_learners.items():
            exp = shap.TreeExplainer(model, feature_perturbation="tree_path_dependent")
            sv  = exp.shap_values(X_df, check_additivity=False)
            if isinstance(sv, list): sv = np.stack(sv, axis=1)
            if sv.ndim == 2: sv = sv[:, np.newaxis, :]
            s, n_ft = sv.shape, len(self.top_10_features)
            if s[1] == n_ft and s[2] == self.n_classes:
                sv = sv.transpose(0,2,1)
            sd[name] = sv
        return sd

    def predict_proba(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.top_10_features)
        X      = X[self.top_10_features]
        Xa     = np.array(X, dtype=float)
        mnames = list(self.base_learners.keys())
        n_mod  = len(mnames)
        probas = np.stack([self.base_learners[n].predict_proba(X) for n in mnames], axis=0)

        wc = self._rank_confidence(probas, n_mod)
        wf = self._shap_alignment(self._compute_shap(X))
        wd = np.tile(self._density(Xa), (n_mod, 1))
        w  = self.alpha*wc + self.beta*wf + self.gamma*wd
        w  = w/np.where(w.sum(axis=0, keepdims=True)<self._EPS, self._EPS, w.sum(axis=0, keepdims=True))
        p  = (probas*w[:,:,np.newaxis]).sum(axis=0)
        p  = p/np.where(p.sum(axis=1, keepdims=True)<self._EPS, self._EPS, p.sum(axis=1, keepdims=True))
        eps = self.label_smooth_eps
        p   = (1-eps)*p + eps/self.n_classes
        return p/p.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.le_classes[np.argmax(self.predict_proba(X), axis=1)]

    def predict_single(self, blood_values: dict) -> dict:
        df = pd.DataFrame([blood_values])[self.top_10_features]
        pr = self.predict_proba(df)[0]
        idx = int(np.argmax(pr))
        return {"disease": str(self.le_classes[idx]),
                "confidence": round(float(pr[idx]), 6),
                "all_probs": {str(c): round(float(p),6) for c,p in zip(self.le_classes,pr)},
                "weights": {"alpha":round(self.alpha,4),"beta":round(self.beta,4),
                            "gamma":round(self.gamma,4)}}

print("Saving CFL model...")
cfl_model = cflModel(
    base_learners=base_learners, knn_purity=knn_purity, knn_scaler=knn_scaler,
    y_train_res=y_train_res, top_10_features=top_10_features, le_classes=le.classes_,
    alpha=ALPHA, beta=BETA, gamma=GAMMA,
    mean_shap_profile=mean_shap_profile, n_classes=n_classes,
    k_neighbors=K_NEIGHBORS, label_smooth_eps=LABEL_SMOOTH,
)
FINAL_PATH = MODELS_DIR+'cfl_model_realworld_v8_final.pkl'
joblib.dump(cfl_model, FINAL_PATH, compress=3)
print(f"✓ Saved → {FINAL_PATH}")
test_in = dict(zip(top_10_features, X_test_sel.iloc[0].values))
print(f"  Verification: {joblib.load(FINAL_PATH).predict_single(test_in)}")

## Cell 22 — Final summary


In [ ]:
print("\n"+"═"*70)
print("  FINAL RESULTS — CFL  (Real-World Imbalanced)")
print("═"*70)
print(pd.DataFrame(results).T.sort_values('ROC_AUC', ascending=False).round(4).to_string())
print("═"*70)
print(f"  α={ALPHA:.4f}  β={BETA:.4f}  γ={GAMMA:.4f}")
print(f"  LABEL_SMOOTH={LABEL_SMOOTH}  |  No isotonic/Platt calibration")
print(f"  kNN manhattan K={K_NEIGHBORS}  |  γ floor ≥ 0.10 enforced")
print(f"  Base learners: {list(base_learners.keys())}")
print("═"*70)

## Cell 23 — Stratified 5-Fold CV


In [ ]:
print("="*65)
print("  STRATIFIED 5-FOLD CV")
print("="*65)

X_cv = np.array(X_data[top_10_features], dtype=float)
y_cv = np.array(y_data)
skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
CV_EPS = 1e-9

# : All 5 base models + Stacking + CFL in CV table
cv_model_specs = {
    'CatBoost':          None,
    'XGBoost':           None,
    'LightGBM':          None,
    'Random Forest':     None,   # FIX 6
    'Extra Trees':       None,   # FIX 6
    'Standard Stacking': None,
    'CFL (Proposed)':  None,
}

cv_scores = {
    name: {'Accuracy':[], 'F1':[], 'ROC_AUC':[], 'Log_Loss':[], 'MCC':[]}
    for name in cv_model_specs
}
fold_labels = np.arange(n_classes)

def score_model(y_true, y_pred, y_prob):
    yp = np.array(y_prob)
    if yp.shape[1] != n_classes:
        yp = yp[:,:n_classes]; yp /= np.where(yp.sum(1,keepdims=True)==0, 1, yp.sum(1,keepdims=True))
    try:
        auc_ = roc_auc_score(y_true, yp, multi_class='ovr', average='macro', labels=fold_labels)
    except ValueError:
        auc_ = np.nan
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1':       f1_score(y_true, y_pred, average='macro', zero_division=0),
        'ROC_AUC':  auc_,
        'Log_Loss': log_loss(y_true, yp, labels=fold_labels),
        'MCC':      matthews_corrcoef(y_true, y_pred),  # FIX 7
    }

for fi, (tr_i, te_i) in enumerate(skf.split(X_cv, y_cv)):
    print(f"\n  Fold {fi+1}/5 ", end='', flush=True)

    Xft = X_cv[tr_i]; Xfe = X_cv[te_i]
    yft = y_cv[tr_i]; yfe = y_cv[te_i]
    Xft_df = pd.DataFrame(Xft, columns=top_10_features)
    Xfe_df = pd.DataFrame(Xfe, columns=top_10_features)

    fold_sc  = StandardScaler()
    Xft_sc   = fold_sc.fit_transform(Xft)
    Xfe_sc   = fold_sc.transform(Xfe)
    xgb_sw_f = compute_sample_weight('balanced', yft)  # FIX 9

    fold_specs = {
        'CatBoost':     CatBoostClassifier(random_state=42, verbose=0, iterations=300,
                            task_type=cat_task, auto_class_weights='Balanced'),
        'XGBoost':      XGBClassifier(random_state=42, eval_metric='mlogloss',
                            n_estimators=200, device=xgb_device, n_jobs=-1),
        'LightGBM':     LGBMClassifier(random_state=42, n_estimators=200,
                            device=lgbm_device, n_jobs=-1, verbose=-1, class_weight='balanced'),
        'Random Forest':RandomForestClassifier(random_state=42, n_estimators=200,
                            n_jobs=-1, class_weight='balanced'),
        'Extra Trees':  ExtraTreesClassifier(random_state=42, n_estimators=200,
                            n_jobs=-1, class_weight='balanced'),
    }

    fold_base = {}
    for name, spec in fold_specs.items():
        m = copy.deepcopy(spec)
        m.fit(Xft_df, yft, sample_weight=xgb_sw_f) if name=='XGBoost' else m.fit(Xft_df, yft)
        fold_base[name] = m

    # Score all 5 base models
    for name in ['CatBoost','XGBoost','LightGBM','Random Forest','Extra Trees']:
        s = score_model(yfe, fold_base[name].predict(Xfe_df), fold_base[name].predict_proba(Xfe_df))
        for metric, val in s.items(): cv_scores[name][metric].append(val)
        print(f"[{name[:3]}✓]", end=' ', flush=True)

    # Standard Stacking in fold
    fold_stk = StackingClassifier(
        estimators=[
            ('cat',  CatBoostClassifier(verbose=0, random_state=42, iterations=300,
                         task_type='CPU', auto_class_weights='Balanced')),
            ('xgb',  XGBClassifier(eval_metric='mlogloss', random_state=42,
                         n_estimators=200, device=xgb_device)),
            ('lgbm', LGBMClassifier(random_state=42, n_estimators=200,
                         device=lgbm_device, verbose=-1, class_weight='balanced')),
        ],
        final_estimator=LogisticRegression(max_iter=2000, class_weight='balanced'),
        stack_method='predict_proba', cv=3, n_jobs=1
    )
    fold_stk.fit(Xft_df, yft)
    s = score_model(yfe, fold_stk.predict(Xfe_df), fold_stk.predict_proba(Xfe_df))
    for metric, val in s.items(): cv_scores['Standard Stacking'][metric].append(val)
    print(f"[Stk✓]", end=' ', flush=True)

    # CFL in fold
    bg_f = []
    for cls in np.unique(yft):
        ci = np.where(yft==cls)[0]
        bg_f.extend(np.random.RandomState(42).choice(
            ci, size=min(SHAP_BG_SIZE//n_classes, len(ci)), replace=False))
    Xbg_f = Xft_df.iloc[bg_f].reset_index(drop=True)

    mi_f  = mutual_info_classif(Xft_df, yft, random_state=42)
    mi_f  = mi_f / (mi_f.sum()+CV_EPS)

    f_shap_test = {}; f_mean_shap = {}
    for name, model in fold_base.items():
        te    = shap.TreeExplainer(model, feature_perturbation="tree_path_dependent")
        sv_bg = te.shap_values(Xbg_f,  check_additivity=False)
        sv_te = te.shap_values(Xfe_df, check_additivity=False)
        if isinstance(sv_bg, list): sv_bg = np.stack(sv_bg,axis=1); sv_te = np.stack(sv_te,axis=1)
        if sv_bg.ndim==2: sv_bg = sv_bg[:,np.newaxis,:]; sv_te = sv_te[:,np.newaxis,:]
        n_ft  = len(top_10_features)
        sv_bg = ensure_shap_shape(sv_bg, n_classes, n_ft)
        sv_te = ensure_shap_shape(sv_te, n_classes, n_ft)
        f_shap_test[name] = sv_te
        mp = np.abs(sv_bg).mean(axis=(0,1)); mp = mp/(mp.sum()+CV_EPS)
        mp = 0.7*mp + 0.3*mi_f; f_mean_shap[name] = mp/(mp.sum()+CV_EPS)

    fknn = KNeighborsClassifier(n_neighbors=K_NEIGHBORS, metric='manhattan', n_jobs=-1)
    fknn.fit(Xft_sc, yft)

    yp_cf, _ = cfl_predict_proba(
        Xfe_df, Xfe_sc, fold_base, f_shap_test, f_mean_shap,
        fknn, yft, n_classes, ALPHA, BETA, GAMMA, K_NEIGHBORS, apply_smooth=True
    )
    s = score_model(yfe, np.argmax(yp_cf, axis=1), yp_cf)
    for metric, val in s.items(): cv_scores['CFL (Proposed)'][metric].append(val)
    print(f"[CFL✓]", end='', flush=True)

print("\n")
print("─"*82)
print(f"  {'Model':<25} {'Accuracy':>13} {'Macro-F1':>13} "
      f"{'ROC-AUC':>13} {'Log-Loss':>13} {'MCC':>13}")
print("─"*82)
cv_summary = {}
for name, scores in cv_scores.items():
    row = {}
    for metric, vals in scores.items():
        a = np.array(vals); row[f'{metric}_mean'] = a.mean(); row[f'{metric}_std'] = a.std()
    cv_summary[name] = row
    marker = " ◄" if name=='CFL (Proposed)' else ""
    print(f"  {name:<25} "
          f"{row['Accuracy_mean']:.4f}±{row['Accuracy_std']:.4f}  "
          f"{row['F1_mean']:.4f}±{row['F1_std']:.4f}  "
          f"{row['ROC_AUC_mean']:.4f}±{row['ROC_AUC_std']:.4f}  "
          f"{row['Log_Loss_mean']:.4f}±{row['Log_Loss_std']:.4f}  "
          f"{row['MCC_mean']:.4f}±{row['MCC_std']:.4f}{marker}")
print("─"*82)
print("  ◄ = Proposed  |  format: mean±std across 5 folds")

## Cell 23b — Cohen's d


In [ ]:
print("\n"+"="*75)
print("  COHEN'S d EFFECT SIZE (5-fold CV)")
print("="*75)

def cohens_d_block(name_a, name_b, label_b):
    print(f"\n  Cohen's d — CFL vs {label_b}:")
    sa = cv_scores[name_a]; sb = cv_scores[name_b]
    for metric in ['Accuracy','F1','ROC_AUC','Log_Loss','MCC']:
        a = np.array(sa[metric]); b = np.array(sb[metric])
        d = abs(a.mean()-b.mean()) / (np.sqrt((a.std()**2+b.std()**2)/2)+CV_EPS)
        interp = ("negligible" if d<0.2 else "small" if d<0.5 else "medium" if d<0.8 else "large")
        print(f"    {metric:<12} d={d:.4f}  ({interp})")

cohens_d_block('CFL (Proposed)', 'Standard Stacking', 'Standard Stacking')
cohens_d_block('CFL (Proposed)', 'Extra Trees',       'Extra Trees')

cv_rows = []
for name, scores in cv_scores.items():
    for fi, vals in enumerate(zip(scores['Accuracy'],scores['F1'],
                                   scores['ROC_AUC'],scores['Log_Loss'],scores['MCC'])):
        cv_rows.append({'Model':name,'Fold':fi+1,'Accuracy':round(vals[0],4),
                        'Macro_F1':round(vals[1],4),'ROC_AUC':round(vals[2],4),
                        'Log_Loss':round(vals[3],4),'MCC':round(vals[4],4)})
cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(RESULTS_DIR+'cv_results_5fold.csv', index=False)
print(f"\n✓ 5-fold CV results saved → {RESULTS_DIR}cv_results_5fold.csv")

fig, axes = plt.subplots(1, 5, figsize=(30, 6))
cv_metrics = ['Accuracy','F1','ROC_AUC','Log_Loss','MCC']
bp_colors  = ['#6baed6','#74c476','#fd8d3c','#9ecae1','#a1d99b','#f768a1','#e6550d']

for ax, metric in zip(axes, cv_metrics):
    data   = [cv_scores[n][metric] for n in cv_model_specs]
    labels = [n.replace(' (Proposed)','\n(★)')
               .replace(' Stacking','\nStacking') for n in cv_model_specs]
    bp = ax.boxplot(data, labels=labels, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], bp_colors):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    bp['boxes'][-1].set_linewidth(2.5); bp['boxes'][-1].set_edgecolor('crimson')
    ax.set_title(metric, fontsize=12); ax.tick_params(axis='x', labelsize=7)
    ax.grid(True, alpha=0.3)
plt.suptitle('5-Fold CV — CFL vs All Baselines (incl. Extra Trees)\n'
             '(Real-World Imbalanced, Macro metrics)', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR+'cv_5fold_boxplot.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print("✓ Boxplot saved.")

## Cell 23c — Friedman Test (Omnibus)


In [ ]:
print("\n" + "="*65)
print("  FRIEDMAN TEST — OMNIBUS MULTI-MODEL COMPARISON")
print("="*65)

from scipy.stats import friedmanchisquare as _friedman
import copy as _copy


print("\n  Generating per-fold scores for DES baselines...")
print("  (Same StratifiedKFold splits as CELL 23 — random_state=42)")

_X_cv = np.array(X_data[top_10_features], dtype=float)
_y_cv = np.array(y_data)
_skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

_des_names = ['KNORA-E', 'KNORA-U', 'META-DES']
_des_cv_scores = {n: {'Accuracy':[], 'F1':[], 'ROC_AUC':[], 'Log_Loss':[], 'MCC':[]}
                  for n in _des_names}

_fold_specs_des = {
    'CatBoost':     CatBoostClassifier(random_state=42, verbose=0, iterations=300,
                        task_type='CPU', auto_class_weights='Balanced'),
    'XGBoost':      XGBClassifier(random_state=42, eval_metric='mlogloss',
                        n_estimators=200, n_jobs=-1),
    'LightGBM':     LGBMClassifier(random_state=42, n_estimators=200,
                        n_jobs=-1, verbose=-1, class_weight='balanced'),
    'Random Forest':RandomForestClassifier(random_state=42, n_estimators=200,
                        n_jobs=-1, class_weight='balanced'),
    'Extra Trees':  ExtraTreesClassifier(random_state=42, n_estimators=200,
                        n_jobs=-1, class_weight='balanced'),
}

for _fi, (_tr_i, _te_i) in enumerate(_skf.split(_X_cv, _y_cv)):
    print(f"  Fold {_fi+1}/5 ", end='', flush=True)
    _Xft = _X_cv[_tr_i]; _Xfe = _X_cv[_te_i]
    _yft = _y_cv[_tr_i]; _yfe = _y_cv[_te_i]
    _Xft_df = pd.DataFrame(_Xft, columns=top_10_features)
    _Xfe_df = pd.DataFrame(_Xfe, columns=top_10_features)
    _xgb_sw = compute_sample_weight('balanced', _yft)

    # Train fold pool
    _fold_pool = []
    for _nm, _sp in _fold_specs_des.items():
        _m = _copy.deepcopy(_sp)
        _m.fit(_Xft_df, _yft, sample_weight=_xgb_sw) if _nm == 'XGBoost' else _m.fit(_Xft_df, _yft)
        _fold_pool.append(_m)

    _Xft_arr = _Xft_df.values
    _Xfe_arr = _Xfe_df.values
    _fold_labels = np.arange(n_classes)

    
    for _des_name, _DESClass, _k_use, _kwargs in [
        ('KNORA-E',  KNORAE,   K_KNORA_E_OPT, {}),
        ('KNORA-U',  KNORAU,   K_KNORA_U_OPT, {}),
        ('META-DES', METADES,  K_META_OPT,    {'Hc': HC_META_OPT}),
    ]:
        try:
            _clf = _DESClass(pool_classifiers=_fold_pool, k=_k_use,
                             DFP=False, with_IH=False, safe_k=5,
                             random_state=42, **_kwargs)
            _clf.fit(_Xft_arr, _yft)
            _yp  = _clf.predict(_Xfe_arr)
            _ypr = _clf.predict_proba(_Xfe_arr)
            if _ypr.shape[1] < n_classes:
                _fp2 = np.zeros((_ypr.shape[0], n_classes))
                _fp2[:, np.unique(_yp)] = _ypr
                _rs2 = _fp2.sum(axis=1, keepdims=True)
                _ypr = _fp2 / np.where(_rs2 < EPS, EPS, _rs2)
                _yp  = np.argmax(_ypr, axis=1)
            try:
                _auc = roc_auc_score(_yfe, _ypr, multi_class='ovr',
                                     average='macro', labels=_fold_labels)
            except ValueError:
                _auc = np.nan
            _s = {
                'Accuracy': accuracy_score(_yfe, _yp),
                'F1':       f1_score(_yfe, _yp, average='macro', zero_division=0),
                'ROC_AUC':  _auc,
                'Log_Loss': log_loss(_yfe, _ypr, labels=_fold_labels),
                'MCC':      matthews_corrcoef(_yfe, _yp),
            }
            for _met, _val in _s.items():
                _des_cv_scores[_des_name][_met].append(_val)
            print(f"[{_des_name[:4]}✓]", end=' ', flush=True)
        except Exception as _ex:
            print(f"[{_des_name[:4]}✗:{_ex}]", end=' ', flush=True)
            for _met in ['Accuracy','F1','ROC_AUC','Log_Loss','MCC']:
                _des_cv_scores[_des_name][_met].append(np.nan)
    print()

# Merge DES cv_scores into main cv_scores dict for Friedman
_cv_scores_full = dict(cv_scores)
_cv_scores_full.update(_des_cv_scores)
print(f"\n✓ cv_scores_full contains {len(_cv_scores_full)} models")

# ── Step 2: Friedman test across all models ───────────────────
_friedman_model_names = list(_cv_scores_full.keys())
_metrics_friedman = {
    'ROC_AUC':  ('ROC-AUC',  True),
    'F1':       ('Macro-F1', True),
    'Log_Loss': ('Log-Loss', False),
    'MCC':      ('MCC',      True),
    'Accuracy': ('Accuracy', True),
}

friedman_results  = {}
friedman_rank_table = {}
friedman_model_names = _friedman_model_names   # expose for CELL 23d

print("\n── Friedman Test Results ────────────────────────────────────")
print(f"  {'Metric':<14} {'χ²':>10}  {'p-value':>10}  Result")
print("  " + "─"*50)

for _mk, (_ml, _hb) in _metrics_friedman.items():
    _mat = []
    for _mn in _friedman_model_names:
        _vals = _cv_scores_full[_mn][_mk]
        _clean = [v if not np.isnan(v) else np.nanmean(_vals) for v in _vals]
        _mat.append(_clean)
    _mat = np.array(_mat)   # (n_models, n_folds)
    try:
        _stat, _p = _friedman(*[_mat[i] for i in range(len(_friedman_model_names))])
        _sig = "SIGNIFICANT ✅" if _p < 0.05 else "Not significant"
        friedman_results[_mk] = {'statistic':_stat,'p_value':_p,
                                  'significant':_p<0.05,'metric_label':_ml}
        print(f"  {_ml:<14} {_stat:>10.4f}  {_p:>10.4f}  {_sig}")

        # Average ranks
        _fold_ranks = np.zeros_like(_mat)
        for _fi in range(_mat.shape[1]):
            _fs = _mat[:, _fi]
            _ord = np.argsort(-_fs if _hb else _fs)
            _rks = np.empty_like(_ord)
            _rks[_ord] = np.arange(1, len(_friedman_model_names)+1)
            _fold_ranks[:, _fi] = _rks
        friedman_rank_table[_mk] = dict(zip(_friedman_model_names, _fold_ranks.mean(axis=1)))
    except Exception as _ex:
        print(f"  {_ml:<14} ERROR: {_ex}")

# Print mean rank table
print("\n── Average Ranks (Rank 1 = best) ───────────────────────────")
_rank_df = pd.DataFrame(friedman_rank_table, index=_friedman_model_names)
_rank_df.columns = [_metrics_friedman[k][0] for k in _rank_df.columns]
_rank_df['Mean Rank'] = _rank_df.mean(axis=1)
_rank_df = _rank_df.sort_values('Mean Rank')
print(_rank_df.round(2).to_string())
_cfl_pos = _rank_df.index.get_loc('CFL (Proposed)') + 1
print(f"\n★ CFL Mean Rank: {_rank_df.loc['CFL (Proposed)','Mean Rank']:.2f} "
      f"(Position {_cfl_pos}/{len(_friedman_model_names)})")

_rank_df.round(4).to_csv(RESULTS_DIR + 'friedman_ranks.csv')
pd.DataFrame([{'Metric':v['metric_label'],'Chi2':round(v['statistic'],4),
               'p_value':round(v['p_value'],4),'Significant':v['significant']}
              for v in friedman_results.values()]).to_csv(
    RESULTS_DIR+'friedman_test.csv', index=False)
print(f"✓ Friedman results saved → {RESULTS_DIR}friedman_test.csv")
print("  → Proceed to Cell 23d for Nemenyi post-hoc + CD Diagram")

## Cell 23d — Nemenyi Post-hoc + Critical Difference Diagram


In [ ]:
print("\n" + "="*65)
print("  NEMENYI POST-HOC + CRITICAL DIFFERENCE DIAGRAM")
print("="*65)

try:
    import scikit_posthocs as _sp2
    print("✓ scikit-posthocs available")
except ImportError:
    import subprocess as _sp3, sys as _sys2
    _sp3.check_call([_sys2.executable, "-m", "pip", "install", "-q", "scikit-posthocs"])
    import scikit_posthocs as _sp2
    print("✓ scikit-posthocs installed")

import matplotlib.patches as _mpatches

# ── CD value (Demšar 2006 Table 5) ───────────────────────────
def _cd_value(n_models, n_folds, alpha=0.05):
    _q = {2:1.960,3:2.343,4:2.569,5:2.728,6:2.850,7:2.949,
          8:3.031,9:3.102,10:3.164,11:3.219,12:3.268}
    q = _q.get(n_models, 3.268)
    return q * np.sqrt(n_models*(n_models+1)/(6*n_folds))

_n_models = len(friedman_model_names)
_n_folds  = 5
_cd       = _cd_value(_n_models, _n_folds)
print(f"  Models: {_n_models} | Folds: {_n_folds} | CD = {_cd:.4f}")

# ── Nemenyi on primary metric: ROC_AUC ───────────────────────
_primary = 'ROC_AUC'
_sm_roc  = np.array([
    [v if not np.isnan(v) else np.nanmean(_cv_scores_full[m][_primary])
     for v in _cv_scores_full[m][_primary]]
    for m in friedman_model_names
])  # (n_models, n_folds)

_nem_roc = _sp2.posthoc_nemenyi_friedman(_sm_roc.T)
_nem_roc.index   = friedman_model_names
_nem_roc.columns = friedman_model_names

print("\n── CFL vs Each Baseline (Nemenyi p-values, ROC-AUC) ─────")
print(f"  {'Baseline':<32}  {'p-value':>10}  {'Sig (α=0.05)?'}")
print("  " + "─"*56)
_cfl_key = 'CFL (Proposed)'
for _mn in friedman_model_names:
    if _mn == _cfl_key: continue
    _p = _nem_roc.loc[_cfl_key, _mn]
    print(f"  {_mn:<32}  {_p:>10.4f}  {'YES ✅' if _p < 0.05 else 'No'}")

# ── Average ranks for CD diagram (ROC_AUC) ───────────────────
_avg_ranks_roc = {}
for _i, _mn in enumerate(friedman_model_names):
    _fr = np.zeros(_n_folds)
    for _fi in range(_n_folds):
        _fs   = _sm_roc[:, _fi]
        _ord  = np.argsort(-_fs)
        _rks  = np.empty_like(_ord)
        _rks[_ord] = np.arange(1, _n_models+1)
        _fr[_fi]   = _rks[_i]
    _avg_ranks_roc[_mn] = _fr.mean()

# ── Draw CD Diagram ───────────────────────────────────────────
def _draw_cd(avg_ranks, cd, model_names, nem_p,
             metric_label, highlight, alpha=0.05, save_path=None):
    _n  = len(model_names)
    _sb = sorted(model_names, key=lambda m: avg_ranks[m])
    fig, ax = plt.subplots(figsize=(14, max(5, _n*0.6+2)))
    ax.set_xlim(0.5, _n+0.5); ax.set_ylim(-2, _n+2.5); ax.axis('off')

    def _mc(name):
        if name == highlight:           return '#e6550d', 2.5, 'bold'
        if 'KNORA' in name or 'META' in name: return '#2171b5', 1.5, 'normal'
        return '#525252', 1.2, 'normal'

    ax.plot([1,_n],[_n+1,_n+1],'k-',lw=1.5)
    for _i in range(1,_n+1):
        ax.plot([_i,_i],[_n+1,_n+0.85],'k-',lw=1.2)
        ax.text(_i,_n+0.55,str(_i),ha='center',va='top',fontsize=9)
    ax.text((_n+1)/2,_n+1.7,f'Average Rank ({metric_label})',
            ha='center',va='bottom',fontsize=11,fontweight='bold')

    for _pos, _mn in enumerate(_sb):
        _r = avg_ranks[_mn]; _y = _n-_pos-0.5
        _c, _lw, _fw = _mc(_mn)
        ax.plot([_r,_r],[_n+1,_y+0.15],color=_c,lw=_lw,ls='--',alpha=0.6)
        ax.text(_n+0.6,_y,_mn,ha='left',va='center',fontsize=8.5,color=_c,fontweight=_fw)
        ax.plot([_r,_n+0.5],[_y,_y],color=_c,lw=0.6,ls=':',alpha=0.4)
        ax.text(_r,_y-0.38,f'{_r:.2f}',ha='center',va='top',fontsize=7.5,color=_c)

    _drawn = []
    for _i, _m1 in enumerate(_sb):
        _grp = [_m1]
        for _m2 in _sb[_i+1:]:
            try:    _pv = nem_p.loc[_m1,_m2]
            except: _pv = 1.0
            if _pv >= alpha: _grp.append(_m2)
        if len(_grp) > 1:
            _rmin = min(avg_ranks[m] for m in _grp)
            _rmax = max(avg_ranks[m] for m in _grp)
            if not any(abs(b[0]-_rmin)<0.01 and abs(b[1]-_rmax)<0.01 for b in _drawn):
                _drawn.append((_rmin,_rmax))
                _yb = -0.2-0.35*len(_drawn)
                ax.plot([_rmin,_rmax],[_yb,_yb],'k-',lw=3.5,
                        solid_capstyle='round',alpha=0.75)

    ax.annotate('',xy=(1.0+cd,_n+1.35),xytext=(1.0,_n+1.35),
                arrowprops=dict(arrowstyle='<->',color='crimson',lw=1.8))
    ax.text(1.0+cd/2,_n+1.55,f'CD={cd:.3f}',ha='center',va='bottom',
            fontsize=9,color='crimson',fontweight='bold')

    _leg = [_mpatches.Patch(color='#e6550d',label='CFL (Proposed)'),
            _mpatches.Patch(color='#2171b5',label='DES Methods (KNORA/META-DES)'),
            _mpatches.Patch(color='#525252',label='Other Baselines'),
            _mpatches.Patch(color='black',  label='Connected = NOT sig. different')]
    ax.legend(handles=_leg,loc='lower left',fontsize=8,frameon=True)
    plt.title(f'Critical Difference Diagram — {metric_label}\n'
              f'Nemenyi post-hoc (α={alpha}, {_n} models, {_n_folds}-fold CV)',
              fontsize=12,fontweight='bold',pad=12)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path,dpi=150,bbox_inches='tight')
        print(f"  ✓ Saved → {save_path}")
    plt.show(); plt.close()

_draw_cd(_avg_ranks_roc, _cd, friedman_model_names, _nem_roc,
         'ROC-AUC', 'CFL (Proposed)',
         save_path=RESULTS_DIR+'cd_diagram_roc_auc.png')

# Also for Macro-F1
_sm_f1 = np.array([
    [v if not np.isnan(v) else np.nanmean(_cv_scores_full[m]['F1'])
     for v in _cv_scores_full[m]['F1']]
    for m in friedman_model_names
])
_nem_f1 = _sp2.posthoc_nemenyi_friedman(_sm_f1.T)
_nem_f1.index = _nem_f1.columns = friedman_model_names
_avg_ranks_f1 = {}
for _i, _mn in enumerate(friedman_model_names):
    _fr = np.zeros(_n_folds)
    for _fi in range(_n_folds):
        _fs = _sm_f1[:, _fi]; _ord = np.argsort(-_fs)
        _rks = np.empty_like(_ord); _rks[_ord] = np.arange(1,_n_models+1)
        _fr[_fi] = _rks[_i]
    _avg_ranks_f1[_mn] = _fr.mean()
_draw_cd(_avg_ranks_f1, _cd, friedman_model_names, _nem_f1,
         'Macro F1', 'CFL (Proposed)',
         save_path=RESULTS_DIR+'cd_diagram_f1.png')

_nem_roc.round(4).to_csv(RESULTS_DIR+'nemenyi_pvalues_roc_auc.csv')
print(f"✓ Nemenyi p-value matrix saved → {RESULTS_DIR}nemenyi_pvalues_roc_auc.csv")
print("✓ Cell 23d complete — CD diagrams (ROC-AUC + F1) produced.")

## Cell 24 — Calibration + ECE


In [ ]:
print("="*65)
print("  RELIABILITY CURVES + ECE (V7 — Publication Quality)")
print("="*65)

# ── ECE function — unchanged from  (downstream cells use this) ─
def compute_ece(y_true, y_prob, n_bins=10):
    """Expected Calibration Error — macro-averaged over classes."""
    n_cls = y_prob.shape[1]
    y_bin = label_binarize(y_true, classes=np.arange(n_cls))
    ece_pc = []
    for c in range(n_cls):
        pc = y_prob[:,c]; tc = y_bin[:,c]
        edges = np.linspace(0,1,n_bins+1); ece_c = 0.0
        for b in range(n_bins):
            mask = (pc>=edges[b])&(pc<edges[b+1])
            if mask.sum()==0: continue
            ece_c += (mask.sum()/len(pc))*abs(tc[mask].mean()-pc[mask].mean())
        ece_pc.append(ece_c)
    return np.mean(ece_pc)

# ── reliability_models — original 5 + KNORA-E + META-DES ────────
# KNORA-E and META-DES added to show their miscalibration visually.
# Original 5 model keys are UNCHANGED — downstream cells safe.
reliability_models = {
    'CatBoost':            base_learners['CatBoost'].predict_proba(X_test_sel),
    'Extra Trees':         base_learners['Extra Trees'].predict_proba(X_test_sel),
    'LightGBM':            base_learners['LightGBM'].predict_proba(X_test_sel),
    'Standard Stacking':   y_prob_stack,
    'KNORA-E':             y_prob_knora_e,
    'META-DES':            y_prob_meta_des,
    'CFL (Proposed)': y_prob_cfl,
}

# ── Distinct style per model (print-safe + greyscale-safe) ───────
_cal_styles = {
    'CatBoost':            ('#4393c3', '--',              'o',  1.5, 5 ),
    'Extra Trees':         ('#f46d43', ':',               's',  1.5, 5 ),
    'LightGBM':            ('#74add1', '-.',              '^',  1.5, 5 ),
    'Standard Stacking':   ('#228b22', (0,(3,1,1,1)),    'D',  1.5, 5 ),
    'KNORA-E':             ('#969696', (0,(5,2)),         'x',  1.5, 6 ),
    'META-DES':            ('#8c6d31', (0,(1,1)),         '+',  1.5, 6 ),
    'CFL (Proposed)': ('#e6550d', '-',               '*',  3.0, 10),
}

# ── Adaptive binning helper ───────────────────────────────────────
def _adaptive_cal_curve(y_true_bin, y_prob_cls, n_bins=5):
    """
    Quantile-based (adaptive) calibration curve for one class.
    Each bin contains equal number of samples — eliminates
    vertical spikes from near-empty high-confidence bins.
    Returns (fraction_of_positives, mean_predicted_prob, bin_counts).
    """
    order      = np.argsort(y_prob_cls)
    sorted_p   = y_prob_cls[order]
    sorted_t   = y_true_bin[order]
    bin_edges  = np.array_split(np.arange(len(sorted_p)), n_bins)
    frac_pos, mean_pred, counts = [], [], []
    for idx in bin_edges:
        if len(idx) == 0: continue
        frac_pos.append(sorted_t[idx].mean())
        mean_pred.append(sorted_p[idx].mean())
        counts.append(len(idx))
    return np.array(frac_pos), np.array(mean_pred), np.array(counts)

# ── Compute ECE and reliability curves ───────────────────────────
y_test_bin = label_binarize(y_test_raw, classes=np.arange(n_classes))
ece_table  = {}

_cal_curves  = {}  # store for plotting
_bin_counts  = {}  # store for histogram

N_BINS_CAL = 5   # adaptive bins — enough for smooth curve, no spikes

print("\nECE — lower is better:")
for mname, yp in reliability_models.items():
    ece = compute_ece(y_test_raw, yp)
    ece_table[mname] = ece

    # Average reliability curve over all classes
    _all_fp, _all_mp, _all_ct = [], [], []
    for c in range(n_classes):
        try:
            fp, mp, ct = _adaptive_cal_curve(
                y_test_bin[:,c], yp[:,c], n_bins=N_BINS_CAL
            )
            _all_fp.append(fp); _all_mp.append(mp); _all_ct.append(ct)
        except Exception:
            pass

    if _all_fp:
        # Bin-wise average across classes
        _fp_arr = np.array(_all_fp)   # (n_classes, n_bins)
        _mp_arr = np.array(_all_mp)
        _ct_arr = np.array(_all_ct)
        _cal_curves[mname]  = (_fp_arr.mean(axis=0), _mp_arr.mean(axis=0))
        _bin_counts[mname]  = _ct_arr.mean(axis=0)

    tag = "◄ Proposed" if 'CFL' in mname else ""
    print(f"  {mname:<40}: ECE={ece:.4f}  {tag}")

# ── Figure: Reliability diagram + histogram ───────────────────────
fig = plt.figure(figsize=(11, 9))
gs  = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.08)
ax_main = fig.add_subplot(gs[0])
ax_hist  = fig.add_subplot(gs[1], sharex=ax_main)

# Perfect calibration diagonal
ax_main.plot([0,1],[0,1],'k--', linewidth=2.0, label='Perfect calibration', zorder=5)

# Plot each model
for mname, (fp, mp) in _cal_curves.items():
    _c, _ls, _mk, _lw, _ms = _cal_styles[mname]
    _label = f"{mname.replace(' (Proposed)','★')}  (ECE={ece_table[mname]:.4f})"
    ax_main.plot(mp, fp,
                 color=_c, linestyle=_ls, linewidth=_lw,
                 marker=_mk, markersize=_ms,
                 label=_label, alpha=0.88, zorder=4 if 'CFL' in mname else 3)

# Shade region between CFL and perfect diagonal
_cfl_fp, _cfl_mp = _cal_curves.get('CFL (Proposed)', (None, None))
if _cfl_fp is not None:
    ax_main.fill_between(_cfl_mp, _cfl_mp, _cfl_fp,
                         alpha=0.10, color='#e6550d', label='CFL calibration gap')

ax_main.set_ylabel('Fraction of Positives', fontsize=12)
ax_main.set_xlim(0, 1); ax_main.set_ylim(0, 1)
ax_main.legend(loc='upper left', fontsize=8.5, framealpha=0.92)
ax_main.grid(True, alpha=0.3)
ax_main.set_title(
    'Reliability Diagram — CFL★ vs All Baselines\n'
    '(Adaptive binning, macro-averaged over classes, incl. KNORA-E & META-DES)',
    fontsize=12, fontweight='bold'
)
plt.setp(ax_main.get_xticklabels(), visible=False)

# Bottom histogram: mean bin sample count (CFL only as reference)
if 'CFL (Proposed)' in _bin_counts:
    _ct = _bin_counts['CFL (Proposed)']
    _mp_cfl = _cal_curves['CFL (Proposed)'][1]
    _bw = np.diff(_mp_cfl).mean() * 0.6 if len(_mp_cfl) > 1 else 0.08
    ax_hist.bar(_mp_cfl, _ct, width=_bw,
                color='#e6550d', alpha=0.6, edgecolor='white')
    ax_hist.set_ylabel('Samples\nper bin', fontsize=9)
    ax_hist.set_xlabel('Mean Predicted Probability', fontsize=12)
    ax_hist.grid(True, alpha=0.3, axis='y')
    ax_hist.tick_params(axis='both', labelsize=9)
    ax_hist.set_xlim(0, 1)

plt.savefig(RESULTS_DIR+'reliability_curve_overall.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f"\n✓ Reliability diagram saved → {RESULTS_DIR}reliability_curve_overall.png")

# ── ECE summary table — same format as  (downstream safe) ──────
ece_df = pd.DataFrame(list(ece_table.items()), columns=['Model','ECE']).sort_values('ECE')
ece_df.to_csv(RESULTS_DIR+'ece_summary.csv', index=False)
print("\nECE Summary (sorted best→worst):")
print(ece_df.to_string(index=False))
print("="*65)

## Cell 24b — Brier Score Decomposition


In [ ]:
print("\n" + "="*65)
print("  BRIER SCORE DECOMPOSITION")
print("="*65)

def _brier_decomp(y_true, y_prob, n_cls, n_bins=10):
    """Murphy (1973) Brier decomposition for multiclass."""
    _n   = len(y_true)
    _lbl = np.arange(n_cls)
    _Y   = label_binarize(y_true, classes=_lbl)
    if _Y.shape[1] == 1: _Y = np.hstack([1-_Y, _Y])
    _bs  = np.mean(np.sum((y_prob - _Y)**2, axis=1))
    _pb  = _Y.mean(axis=0)
    _unc = np.sum(_pb*(1-_pb))
    _rel = 0.0; _res = 0.0
    for _k in range(n_cls):
        _pk = y_prob[:,_k]; _lk = _Y[:,_k]; _pbk = _lk.mean()
        _be = np.linspace(0,1,n_bins+1)
        _bi = np.clip(np.digitize(_pk,_be,right=True),1,n_bins)
        for _b in range(1,n_bins+1):
            _mk = _bi==_b; _nb = _mk.sum()
            if _nb == 0: continue
            _fb = _pk[_mk].mean(); _ob = _lk[_mk].mean()
            _rel += _nb*(_fb-_ob)**2
            _res += _nb*(_ob-_pbk)**2
    _rel /= (_n*n_cls); _res /= (_n*n_cls)
    return {'Brier_Score':round(_bs,4),'Reliability':round(_rel,4),
            'Resolution':round(_res,4),'Uncertainty':round(_unc,4)}

# Build model probability dict from existing scope variables
_brier_probs = {
    'CFL (Proposed)':     y_prob_cfl,
    'CatBoost':             base_learners['CatBoost'].predict_proba(X_test_sel),
    'XGBoost':              base_learners['XGBoost'].predict_proba(X_test_sel),
    'LightGBM':             base_learners['LightGBM'].predict_proba(X_test_sel),
    'Random Forest':        base_learners['Random Forest'].predict_proba(X_test_sel),
    'Extra Trees':          base_learners['Extra Trees'].predict_proba(X_test_sel),
    'Std. Stacking':        y_prob_stack,
    'KNORA-E':              y_prob_knora_e,
    'KNORA-U':              y_prob_knora_u,
    'META-DES':             y_prob_meta_des,
}

_brier_res = {nm: _brier_decomp(y_test_raw, yp, n_classes)
              for nm, yp in _brier_probs.items()}

_brier_df = pd.DataFrame(_brier_res).T.sort_values('Brier_Score')
print(f"\n  {'Model':<25} {'Brier↓':>8} {'Reliability↓':>13} "
      f"{'Resolution↑':>12} {'Uncertainty':>12}")
print("  "+"─"*72)
for _mn, _row in _brier_df.iterrows():
    _mk = " ◄" if 'CFL' in _mn else ""
    print(f"  {_mn:<25} {_row['Brier_Score']:>8.4f} {_row['Reliability']:>13.4f} "
          f"{_row['Resolution']:>12.4f} {_row['Uncertainty']:>12.4f}{_mk}")
print("  Note: Uncertainty is dataset-level — identical for all models.")

# Plot
_fig, _axs = plt.subplots(1,2,figsize=(18,7))
_mo = _brier_df.index.tolist()
_xp = np.arange(len(_mo))
_rv = _brier_df['Reliability'].values
_sv = _brier_df['Resolution'].values
_clrs = ['crimson' if 'CFL' in m else
         '#2171b5' if ('KNORA' in m or 'META' in m) else '#969696' for m in _mo]

_axs[0].bar(_xp, _rv, 0.55, label='Reliability (↓ better)', color='#d94701', alpha=0.8)
_axs[0].bar(_xp, _sv, 0.55, bottom=_rv, label='Resolution (↑ better)', color='#2171b5', alpha=0.8)
_axs[0].set_xticks(_xp)
_axs[0].set_xticklabels([m.replace(' ','\\n') for m in _mo], fontsize=8, rotation=15)
_axs[0].set_title('Reliability + Resolution Decomposition', fontsize=11, fontweight='bold')
_axs[0].legend(fontsize=9); _axs[0].grid(True,axis='y',alpha=0.3)

_bv = _brier_df['Brier_Score'].values
_br = _axs[1].bar(_xp, _bv, 0.55, color=_clrs, alpha=0.85, edgecolor='white')
for _b, _v in zip(_br, _bv):
    _axs[1].text(_b.get_x()+_b.get_width()/2, _b.get_height()+0.002,
                 f'{_v:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
_axs[1].set_xticks(_xp)
_axs[1].set_xticklabels([m.replace(' ','\\n') for m in _mo], fontsize=8, rotation=15)
_axs[1].set_title('Total Brier Score (↓ better)', fontsize=11, fontweight='bold')
_axs[1].grid(True,axis='y',alpha=0.3)

from matplotlib.patches import Patch as _Patch
_axs[1].legend(handles=[_Patch(color='crimson',label='CFL'),
                         _Patch(color='#2171b5',label='DES Methods'),
                         _Patch(color='#969696',label='Other')], fontsize=9)
plt.suptitle('Brier Score Decomposition — Calibration Mechanism Analysis',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR+'brier_decomposition.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
_brier_df.to_csv(RESULTS_DIR+'brier_decomposition.csv')
print(f"✓ Brier decomposition saved → {RESULTS_DIR}brier_decomposition.png / .csv")

## Cell 24c — Rejection Curve (Selective Prediction)


In [ ]:
print("\n" + "="*65)
print("  REJECTION CURVE (SELECTIVE PREDICTION ANALYSIS)")
print("="*65)

def _rejection_curve(y_true, y_prob, n_pts=100):
    _n     = len(y_true)
    _unc   = 1 - y_prob.max(axis=1)
    _ypred = y_prob.argmax(axis=1)
    _corr  = (_ypred == y_true).astype(float)
    _ord   = np.argsort(_unc)
    _cs    = _corr[_ord]
    _fracs = np.linspace(0, 0.95, n_pts)
    _accs  = []
    for _f in _fracs:
        _nk = max(1, int(_n*(1-_f)))
        _accs.append(_cs[:_nk].mean())
    _accs = np.array(_accs)
    _auc  = np.trapz(_accs, _fracs) / (_fracs[-1]-_fracs[0])
    return _fracs, _accs, _auc

_rc_probs = {
    'CFL (Proposed)': y_prob_cfl,
    'CatBoost':         base_learners['CatBoost'].predict_proba(X_test_sel),
    'Extra Trees':      base_learners['Extra Trees'].predict_proba(X_test_sel),
    'Std. Stacking':    y_prob_stack,
    'KNORA-E':          y_prob_knora_e,
    'KNORA-U':          y_prob_knora_u,
    'META-DES':         y_prob_meta_des,
}

_rc_results = {}
print(f"  {'Model':<25}  AUC-RC")
print("  "+"─"*35)
for _nm, _yp in _rc_probs.items():
    _f, _a, _au = _rejection_curve(y_test_raw, _yp)
    _rc_results[_nm] = {'fracs':_f,'accs':_a,'auc_rc':_au}
    _mk = " ◄" if 'CFL' in _nm else ""
    print(f"  {_nm:<25}  {_au:.4f}{_mk}")

def _rc_style(name):
    if 'CFL' in name:    return '#e6550d', 2.5, '-'
    if 'META' in name:     return '#2171b5', 1.8, '--'
    if 'KNORA' in name:    return '#6baed6', 1.5, '--'
    if 'Stacking' in name: return '#74c476', 1.5, '-.'
    return '#bdbdbd', 1.2, ':'

_fig2, _ax2s = plt.subplots(1,2,figsize=(18,7))
for _nm, _rc in _rc_results.items():
    _c,_lw,_ls = _rc_style(_nm)
    _ax2s[0].plot(_rc['fracs']*100, _rc['accs']*100,
                  label=f"{_nm} (AUC={_rc['auc_rc']:.3f})",
                  color=_c, lw=_lw, ls=_ls)
    _msk = _rc['fracs'] <= 0.30
    _ax2s[1].plot(_rc['fracs'][_msk]*100, _rc['accs'][_msk]*100,
                  label=_nm, color=_c, lw=_lw, ls=_ls)

for _ax_ in _ax2s:
    _ax_.grid(True, alpha=0.3)
    _ax_.set_xlabel('Rejection Rate (%)', fontsize=11)
    _ax_.set_ylabel('Accuracy on Accepted Samples (%)', fontsize=11)
    _ax_.legend(fontsize=8, loc='upper left')
_ax2s[0].set_title('Rejection Curve — Full Range (0–95%)', fontsize=11, fontweight='bold')
_ax2s[0].set_xlim(0,95)
_ax2s[1].set_title('Rejection Curve — Clinical Range (0–30%)', fontsize=11, fontweight='bold')
_ax2s[1].set_xlim(0,30)

plt.suptitle('Rejection Curve — Selective Prediction Quality\n'
             'Real-World Imbalanced Blood Disease Dataset',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR+'rejection_curve.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

pd.DataFrame([{'Model':m,'AUC_RC':round(_rc_results[m]['auc_rc'],4)}
              for m in _rc_results]).sort_values('AUC_RC',ascending=False
             ).to_csv(RESULTS_DIR+'rejection_curve_auc.csv', index=False)
print(f"✓ Rejection curve saved → {RESULTS_DIR}rejection_curve.png / rejection_curve_auc.csv")

## Cell 25 — Weight Interpretability


In [ ]:
print("\n" + "="*65)
print("  WEIGHT INTERPRETABILITY")
print("="*65)

# sample_weights shape: (n_test, n_learners) — from CELL 14
# model_names_list     — from CELL 16
_sw   = sample_weights   # (n_test, 5)
_nl   = len(model_names_list)
_eq_w = 1.0 / _nl
print(f"✓ sample_weights shape: {_sw.shape}")
print(f"  Base learners: {model_names_list}")
print(f"  Equal weight baseline: {_eq_w:.4f}")

# ── Figure 1: Mean ± std per learner ─────────────────────────
_fig3, _ax3s = plt.subplots(1,2,figsize=(18,6))

_mw  = _sw.mean(axis=0)
_sw_ = _sw.std(axis=0)
_bc  = ['#e6550d','#2171b5','#31a354','#fd8d3c','#756bb1']
_br3 = _ax3s[0].bar(range(_nl), _mw, color=_bc, alpha=0.82,
                     edgecolor='white', width=0.6)
_ax3s[0].errorbar(range(_nl), _mw, yerr=_sw_,
                  fmt='none', color='black', capsize=5, lw=1.5)
for _b, _v in zip(_br3, _mw):
    _ax3s[0].text(_b.get_x()+_b.get_width()/2,
                  _b.get_height()+_sw_[list(_mw).index(_v)]+0.005,
                  f'{_v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
_ax3s[0].axhline(y=_eq_w, color='grey', ls='--', lw=1.5, label=f'Equal weight={_eq_w:.3f}')
_ax3s[0].set_xticks(range(_nl))
_ax3s[0].set_xticklabels(model_names_list, fontsize=9, rotation=15)
_ax3s[0].set_ylabel('Mean Combined Weight (± std)', fontsize=10)
_ax3s[0].set_title('Mean Final Weight per Base Learner\n(Averaged over all test samples)',
                   fontsize=11, fontweight='bold')
_ax3s[0].legend(fontsize=9); _ax3s[0].grid(True,axis='y',alpha=0.3)

# ── Figure 2: Per-class heatmap ───────────────────────────────
_wbc = np.zeros((n_classes, _nl))
for _c in range(n_classes):
    _mask = y_test_raw == _c
    if _mask.sum() > 0:
        _wbc[_c] = _sw[_mask].mean(axis=0)

_im = _ax3s[1].imshow(_wbc, aspect='auto', cmap='YlOrRd')
_ax3s[1].set_xticks(range(_nl))
_ax3s[1].set_xticklabels(model_names_list, fontsize=9, rotation=20, ha='right')
_ax3s[1].set_yticks(range(n_classes))
_ax3s[1].set_yticklabels([f'Class {i}' for i in range(n_classes)], fontsize=8)
_ax3s[1].set_title('Mean Weight per Learner by True Class\n(Shows class-adaptive weighting)',
                   fontsize=11, fontweight='bold')
for _i in range(n_classes):
    for _j in range(_nl):
        _ax3s[1].text(_j,_i,f'{_wbc[_i,_j]:.3f}',ha='center',va='center',
                      fontsize=7.5,
                      color='white' if _wbc[_i,_j]>0.25 else 'black')
plt.colorbar(_im, ax=_ax3s[1], label='Mean Weight')

plt.suptitle('CFL Dynamic Weight Interpretability\n'
             'How ensemble weighting adapts per sample and per class',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR+'weight_interpretability.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

# ── Figure 3: Weight concentration vs confidence ──────────────
_conf = y_prob_cfl.max(axis=1)
_wcon = _sw.max(axis=1)
_fig4, _ax4 = plt.subplots(figsize=(10,6))
_sc4 = _ax4.scatter(_conf, _wcon, c=_conf, cmap='RdYlGn', alpha=0.5, s=15)
plt.colorbar(_sc4, ax=_ax4, label='CFL Confidence (max prob)')
_z4 = np.polyfit(_conf, _wcon, 1)
_p4 = np.poly1d(_z4)
_xl = np.linspace(_conf.min(), _conf.max(), 100)
_ax4.plot(_xl, _p4(_xl), 'r--', lw=2, label=f'Trend (slope={_z4[0]:.3f})')
_ax4.set_xlabel('CFL Confidence (max predicted probability)', fontsize=11)
_ax4.set_ylabel('Weight Concentration (max learner weight)', fontsize=11)
_ax4.set_title('Weight Concentration vs Model Confidence\n'
               'High confidence → more concentrated weights',
               fontsize=11, fontweight='bold')
_ax4.legend(fontsize=10); _ax4.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR+'weight_vs_confidence.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

# ── Summary stats ─────────────────────────────────────────────
print("\n── Weight Summary Statistics ───────────────────────────────")
print(f"  {'Learner':<18} {'Mean W':>8} {'Std W':>8} {'> Equal?':>10}")
print("  "+"─"*46)
for _j, _nm in enumerate(model_names_list):
    _w  = _sw[:, _j]
    _ab = (_w > _eq_w).mean()*100
    print(f"  {_nm:<18} {_w.mean():>8.4f} {_w.std():>8.4f} {_ab:>9.1f}%")

np.save(RESULTS_DIR+'weight_matrix.npy', _sw)
print(f"\n✓ Weight interpretability figures saved → {RESULTS_DIR}")
print(f"  weight_interpretability.png | weight_vs_confidence.png")
print(f"  weight_matrix.npy")
print("✓ Cell 25 complete.")

## Cell 25b — Per-Class F1 Breakdown (Extended)


In [ ]:
print("\n" + "="*65)
print("  PER-CLASS F1 BREAKDOWN (EXTENDED — incl. DES Baselines)")
print("="*65)

# ── Build prediction dict — all 9 models ─────────────────────
# Base learners predict on unscaled X_test_sel (tree-safe)
_pc_preds = {
    'CatBoost':         base_learners['CatBoost'].predict(X_test_sel),
    'XGBoost':          base_learners['XGBoost'].predict(X_test_sel),
    'LightGBM':         base_learners['LightGBM'].predict(X_test_sel),
    'Random Forest':    base_learners['Random Forest'].predict(X_test_sel),
    'Extra Trees':      base_learners['Extra Trees'].predict(X_test_sel),
    'Std. Stacking':    y_pred_stack,
    'KNORA-E':          y_pred_knora_e,
    'KNORA-U':          y_pred_knora_u,
    'META-DES':         y_pred_meta_des,
    'CFL (Proposed)': y_pred_cfl,
}

# ── Compute per-class F1 for each model ──────────────────────
_f1_pc = {
    mn: f1_score(y_test_raw, yp, average=None,
                 labels=class_labels, zero_division=0)
    for mn, yp in _pc_preds.items()
}

# DataFrame: rows = models, columns = disease class names
_f1_df_ext = pd.DataFrame(_f1_pc, index=disease_names).T
_f1_df_ext.index.name = 'Model'

# ── Print table ───────────────────────────────────────────────
print(f"\n── Per-Class F1 Score (all {n_classes} classes) ─────────────────")
print(_f1_df_ext.round(4).to_string())

# ── CFL win/loss per class ──────────────────────────────────
print(f"\n── CFL vs Each Baseline Per Class ────────────────────────")
print(f"  (↑ = CFL wins, ↓ = CFL loses, = = tied)")
_cfl_f1 = _f1_df_ext.loc['CFL (Proposed)'].values

print(f"\n  {'Model':<20}", end='')
for _dn in disease_names:
    print(f"  {_dn[:10]:>10}", end='')
print()
print("  " + "─"*(22 + 13*len(disease_names)))

for _mn in _f1_df_ext.index:
    if _mn == 'CFL (Proposed)': continue
    _baseline_f1 = _f1_df_ext.loc[_mn].values
    print(f"  {_mn:<20}", end='')
    for _cfv, _bsv in zip(_cfl_f1, _baseline_f1):
        _diff = _cfv - _bsv
        if abs(_diff) < 0.005:   _sym = '  ='
        elif _diff > 0:          _sym = f' ↑{_diff:+.3f}'[:-1] if False else f'↑{_diff:.3f}'
        else:                    _sym = f'↓{abs(_diff):.3f}'
        print(f"  {_sym:>10}", end='')
    print()

# ── Count wins per class ──────────────────────────────────────
print(f"\n── CFL Win Count per Class ───────────────────────────────")
print(f"  (How many baselines does CFL beat on each disease?)")
_n_baselines = len(_f1_df_ext) - 1   # exclude CFL itself
for _ci, _dn in enumerate(disease_names):
    _cfv = _cfl_f1[_ci]
    _wins = sum(1 for _mn in _f1_df_ext.index
                if _mn != 'CFL (Proposed)'
                and _cfv > _f1_df_ext.loc[_mn, _dn] + 0.005)
    _bar = '█' * _wins + '░' * (_n_baselines - _wins)
    # Also get class support (sample count)
    _support = int((y_test_raw == class_labels[_ci]).sum())
    print(f"  {_dn:<23} [{_bar}] {_wins}/{_n_baselines}  "
          f"(CFL F1={_cfv:.3f}, n={_support})")

# ── Figure 1: Grouped bar chart ───────────────────────────────
_plot_models = [
    'Extra Trees', 'Std. Stacking',
    'KNORA-E', 'KNORA-U', 'META-DES',
    'CFL (Proposed)'
]
_plot_colors = {
    'Extra Trees':      '#fd8d3c',
    'Std. Stacking':    '#74c476',
    'KNORA-E':          '#969696',
    'KNORA-U':          '#bdbdbd',
    'META-DES':         '#8c6d31',
    'CFL (Proposed)': '#e6550d',
}

_f1_plot = _f1_df_ext.loc[_plot_models]
_x       = np.arange(len(disease_names))
_n_plot  = len(_plot_models)
_width   = 0.13
_offsets = np.linspace(-(_n_plot-1)/2, (_n_plot-1)/2, _n_plot) * _width

fig, ax = plt.subplots(figsize=(16, 7))
for _i, (_mn, _off) in enumerate(zip(_plot_models, _offsets)):
    _vals = _f1_plot.loc[_mn].values
    _bars = ax.bar(_x + _off, _vals, _width,
                   color=_plot_colors[_mn],
                   label=_mn,
                   edgecolor='white',
                   linewidth=0.5,
                   alpha=0.9 if 'CFL' in _mn else 0.75,
                   zorder=3 if 'CFL' in _mn else 2)
    # Bold outline for CFL bars
    if 'CFL' in _mn:
        for _bar in _bars:
            _bar.set_edgecolor('crimson')
            _bar.set_linewidth(1.8)

# Class support annotations on x-axis
_xtick_labels = []
for _ci, _dn in enumerate(disease_names):
    _support = int((y_test_raw == class_labels[_ci]).sum())
    _xtick_labels.append(f"{_dn}\n(n={_support})")

ax.set_xticks(_x)
ax.set_xticklabels(_xtick_labels, fontsize=9)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_title(
    'Per-Class F1 Score — CFL vs Key Baselines\n'
    '(incl. KNORA-E, KNORA-U, META-DES | n = test set support per class)',
    fontsize=12, fontweight='bold'
)
ax.legend(loc='upper right', fontsize=9, framealpha=0.92)
ax.grid(True, axis='y', alpha=0.3, zorder=0)
ax.axhline(y=0, color='black', linewidth=0.8)

# Macro-F1 reference line for CFL
_cfl_macro = _f1_df_ext.loc['CFL (Proposed)'].mean()
ax.axhline(y=_cfl_macro, color='#e6550d', linewidth=1.2,
           linestyle='--', alpha=0.6,
           label=f'CFL Macro-F1={_cfl_macro:.3f}')

plt.tight_layout()
plt.savefig(RESULTS_DIR + 'per_class_f1_extended.png',
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()

# ── Figure 2: Heatmap — easy to read in papers ───────────────
fig2, ax2 = plt.subplots(figsize=(14, 5))
_hm_data = _f1_df_ext.copy()

# Highlight CFL row
import matplotlib.colors as _mcolors
_cmap = plt.cm.RdYlGn   # red=low, green=high

_im = ax2.imshow(_hm_data.values, aspect='auto',
                 cmap=_cmap, vmin=0, vmax=1)

ax2.set_xticks(range(len(disease_names)))
ax2.set_xticklabels(disease_names, fontsize=9, rotation=25, ha='right')
ax2.set_yticks(range(len(_hm_data)))
ax2.set_yticklabels(_hm_data.index, fontsize=9)

# Text annotations
for _i in range(len(_hm_data)):
    for _j in range(len(disease_names)):
        _val = _hm_data.iloc[_i, _j]
        _tc  = 'white' if _val < 0.3 or _val > 0.8 else 'black'
        _fw  = 'bold' if _hm_data.index[_i] == 'CFL (Proposed)' else 'normal'
        ax2.text(_j, _i, f'{_val:.3f}',
                 ha='center', va='center',
                 fontsize=8, color=_tc, fontweight=_fw)

# Bold box around CFL row
_cfl_row_idx = list(_hm_data.index).index('CFL (Proposed)')
_rect = plt.Rectangle(
    (-0.5, _cfl_row_idx - 0.5),
    len(disease_names), 1,
    linewidth=2.5, edgecolor='crimson', facecolor='none'
)
ax2.add_patch(_rect)

plt.colorbar(_im, ax=ax2, label='F1-Score', fraction=0.02)
ax2.set_title(
    'Per-Class F1 Heatmap — All Models\n'
    '(CFL row highlighted in red border)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'per_class_f1_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()

# ── Save ──────────────────────────────────────────────────────
_f1_df_ext.round(4).to_csv(RESULTS_DIR + 'per_class_f1_extended.csv')
print(f"\n✓ Per-class F1 bar chart saved → {RESULTS_DIR}per_class_f1_extended.png")
print(f"✓ Per-class F1 heatmap saved  → {RESULTS_DIR}per_class_f1_heatmap.png")
print(f"✓ Per-class F1 CSV saved      → {RESULTS_DIR}per_class_f1_extended.csv")
print("✓ Cell 25b complete.")

## Cell 25c — DET Curve (Detection Error Tradeoff)


In [ ]:
print("\n" + "="*65)
print("  DET CURVE — DETECTION ERROR TRADEOFF ANALYSIS")
print("="*65)

# ── Import det_curve (not in CELL 3 imports) ─────────────────
from sklearn.metrics import det_curve

# ── Model probability dict ────────────────────────────────────
# All probability arrays extracted from existing scope.
# No recomputation of CFL or any trained model.
_det_probs = {
    'CatBoost':         base_learners['CatBoost'].predict_proba(X_test_sel),
    'Extra Trees':      base_learners['Extra Trees'].predict_proba(X_test_sel),
    'Std. Stacking':    y_prob_stack,
    'KNORA-E':          y_prob_knora_e,
    'KNORA-U':          y_prob_knora_u,
    'META-DES':         y_prob_meta_des,
    'CFL (Proposed)': y_prob_cfl,
}

# ── Distinct styles — same convention as reliability diagram ──
_det_styles = {
    'CatBoost':         ('#4393c3', '--',           1.5),
    'Extra Trees':      ('#f46d43', ':',            1.5),
    'Std. Stacking':    ('#228b22', (0,(3,1,1,1)), 1.5),
    'KNORA-E':          ('#969696', (0,(5,2)),      1.5),
    'KNORA-U':          ('#bdbdbd', (0,(4,2,1,2)), 1.5),
    'META-DES':         ('#8c6d31', (0,(1,1)),      1.5),
    'CFL (Proposed)': ('#e6550d', '-',            3.0),
}

# ── Helper: macro-averaged DET curve ─────────────────────────
def _macro_det(y_true_bin, y_prob, n_cls):
    """
    Compute macro-averaged DET curve.
    Interpolates each class FPR/FNR onto a common FPR grid,
    then averages FNR across all classes.

    Returns (fpr_grid, mean_fnr, auc_det)
    auc_det = area under DET curve (lower is better).
    """
    fpr_grid = np.linspace(0, 1, 500)
    fnr_interp = np.zeros((n_cls, len(fpr_grid)))

    for c in range(n_cls):
        try:
            fpr_c, fnr_c, _ = det_curve(y_true_bin[:, c], y_prob[:, c])
            # det_curve goes from high threshold to low
            # ensure fpr is monotonically increasing for interp
            sort_idx = np.argsort(fpr_c)
            fpr_c = fpr_c[sort_idx]
            fnr_c = fnr_c[sort_idx]
            fnr_interp[c] = np.interp(fpr_grid, fpr_c, fnr_c)
        except Exception:
            fnr_interp[c] = np.ones(len(fpr_grid))

    mean_fnr = fnr_interp.mean(axis=0)
    auc_det  = np.trapz(mean_fnr, fpr_grid)
    return fpr_grid, mean_fnr, auc_det

# ── Compute macro DET for all models ─────────────────────────
print("\n  Macro-averaged DET curves:")
print(f"  {'Model':<25}  AUC-DET (↓ better)")
print("  " + "─"*42)

_det_results = {}
for _mn, _yp in _det_probs.items():
    _fpr, _fnr, _auc = _macro_det(y_test_bin, _yp, n_classes)
    _det_results[_mn] = {'fpr': _fpr, 'fnr': _fnr, 'auc_det': _auc}
    _mk = " ◄" if 'CFL' in _mn else ""
    print(f"  {_mn:<25}  {_auc:.4f}{_mk}")

# ── Figure 1: Macro DET curve — all models ───────────────────
fig1, ax1 = plt.subplots(figsize=(10, 8))

# Random classifier reference line (FNR = 1 - FPR)
_fpr_ref = np.linspace(0, 1, 200)
ax1.plot(_fpr_ref, 1 - _fpr_ref, 'k:', linewidth=1.2,
         alpha=0.5, label='Random classifier')

for _mn, _dr in _det_results.items():
    _c, _ls, _lw = _det_styles[_mn]
    _label = f"{_mn}  (AUC={_dr['auc_det']:.4f})"
    ax1.plot(_dr['fpr'], _dr['fnr'],
             color=_c, linestyle=_ls, linewidth=_lw,
             label=_label, alpha=0.88,
             zorder=4 if 'CFL' in _mn else 3)

# Shade clinical operating region (FPR < 0.2)
ax1.axvspan(0, 0.2, alpha=0.06, color='green',
            label='Clinical operating region (FPR < 0.2)')
ax1.axvline(x=0.2, color='green', linewidth=1.0,
            linestyle='--', alpha=0.5)

ax1.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax1.set_ylabel('False Negative Rate (FNR)', fontsize=12)
ax1.set_title(
    'DET Curve — Macro-Averaged over All Disease Classes\n'
    'CFL vs All Baselines (incl. KNORA-E, KNORA-U, META-DES)\n'
    'Lower curve = better | Shaded = clinical operating region',
    fontsize=11, fontweight='bold'
)
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
ax1.legend(loc='upper right', fontsize=8.5, framealpha=0.92)
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'det_curve_macro.png',
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f"\n✓ Macro DET curve saved → {RESULTS_DIR}det_curve_macro.png")

# ── Figure 2: Per-class DET curves ───────────────────────────
# One subplot per disease class.
# Key models only: CFL, Extra Trees, KNORA-E, META-DES.
# Keeps figure readable.
_key_models_det = {
    'Extra Trees':      _det_styles['Extra Trees'],
    'KNORA-E':          _det_styles['KNORA-E'],
    'META-DES':         _det_styles['META-DES'],
    'CFL (Proposed)': _det_styles['CFL (Proposed)'],
}
_key_probs_det = {m: _det_probs[m] for m in _key_models_det}

_ncols = 3
_nrows = int(np.ceil(n_classes / _ncols))
fig2, axes2 = plt.subplots(_nrows, _ncols,
                            figsize=(_ncols * 5, _nrows * 4.5))
axes2 = axes2.flatten()

for _ci in range(n_classes):
    ax = axes2[_ci]
    _dn = disease_names[_ci]
    _support = int((y_test_raw == class_labels[_ci]).sum())

    # Random reference
    ax.plot([0, 1], [1, 0], 'k:', linewidth=1.0,
            alpha=0.4, label='Random')

    for _mn, (_c, _ls, _lw) in _key_models_det.items():
        _yp = _key_probs_det[_mn]
        try:
            _fpr_c, _fnr_c, _ = det_curve(
                y_test_bin[:, _ci], _yp[:, _ci]
            )
            # Sort by FPR for clean interpolation
            _si = np.argsort(_fpr_c)
            ax.plot(_fpr_c[_si], _fnr_c[_si],
                    color=_c, linestyle=_ls, linewidth=_lw,
                    label=_mn, alpha=0.88,
                    zorder=4 if 'CFL' in _mn else 3)
        except Exception as _ex:
            print(f"  ⚠ {_mn} class {_dn}: {_ex}")

    # Clinical region shading
    ax.axvspan(0, 0.2, alpha=0.06, color='green')

    ax.set_title(f'{_dn}\n(n={_support})',
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('FPR', fontsize=8)
    ax.set_ylabel('FNR', fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.25)

    # Legend only on first subplot
    if _ci == 0:
        ax.legend(fontsize=7, loc='upper right', framealpha=0.9)

# Hide unused subplots
for _ci in range(n_classes, len(axes2)):
    axes2[_ci].set_visible(False)

plt.suptitle(
    'Per-Class DET Curves — CFL vs Key Baselines\n'
    'Each subplot = one disease class | '
    'Green shading = clinical operating region (FPR < 0.2)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'det_curve_per_class.png',
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f"✓ Per-class DET curve saved → {RESULTS_DIR}det_curve_per_class.png")

# ── AUC-DET summary table ─────────────────────────────────────
print("\n── AUC-DET Summary (lower = better FNR/FPR tradeoff) ──────")
_auc_det_df = pd.DataFrame([{
    'Model':   _mn,
    'AUC_DET': round(_det_results[_mn]['auc_det'], 4),
}for _mn in _det_results]).sort_values('AUC_DET')

for _, _row in _auc_det_df.iterrows():
    _mk = " ◄" if 'CFL' in _row['Model'] else ""
    print(f"  {_row['Model']:<25}  AUC-DET = {_row['AUC_DET']:.4f}{_mk}")

_auc_det_df.to_csv(RESULTS_DIR + 'det_curve_auc.csv', index=False)
print(f"\n✓ AUC-DET table saved → {RESULTS_DIR}det_curve_auc.csv")

# ── Interpretation ────────────────────────────────────────────
print("\n── Interpretation ──────────────────────────────────────────")
_cfl_auc = _det_results['CFL (Proposed)']['auc_det']
_best_auc  = min(_det_results[_mn]['auc_det'] for _mn in _det_results)
_best_mn   = min(_det_results, key=lambda m: _det_results[m]['auc_det'])

if _best_mn == 'CFL (Proposed)':
    print("  ✅ CFL has the LOWEST AUC-DET — best FNR/FPR tradeoff")
    print("     → Most useful for clinical triage where missing a")
    print("       disease (FNR) is more costly than a false alarm.")
else:
    _delta = _cfl_auc - _best_auc
    print(f"  ⚠ {_best_mn} has lower AUC-DET by {_delta:.4f}")
    print(f"    CFL: {_cfl_auc:.4f} | Best: {_best_auc:.4f}")
    print("    Frame: CFL's calibration advantage (ECE, Log-Loss)")
    print("    complements the DET analysis — the two measure")
    print("    different aspects of clinical utility.")

print("\n✓ Cell 25c complete.")
print("  → 2 figures: det_curve_macro.png + det_curve_per_class.png")
print("  → 1 CSV:     det_curve_auc.csv")

## Cell 26 — Computational Cost Analysis


In [ ]:
print("\n" + "="*65)
print("  COMPUTATIONAL COST ANALYSIS")
print("="*65)

import time
import tracemalloc
import copy as _copy2
import warnings
warnings.filterwarnings('ignore')

_N_INFERENCE_RUNS = 10   # median over 10 runs for stable estimate
_n_test           = len(X_test_sel)
_n_train          = len(X_train_res)

print(f"\n  Train set : {_n_train} samples")
print(f"  Test set  : {_n_test}  samples")
print(f"  Inference : median of {_N_INFERENCE_RUNS} runs\n")

# ── Helper: measure inference time + peak memory ─────────────
def _measure_inference(predict_fn, n_runs=10):
    """
    Returns (median_ms_per_sample, peak_memory_mb).
    Runs predict_fn() n_runs times, records wall time each run.
    Uses tracemalloc for peak memory on the first run only.
    """
    times = []
    for i in range(n_runs):
        if i == 0:
            tracemalloc.start()
        _t0 = time.perf_counter()
        predict_fn()
        _t1 = time.perf_counter()
        if i == 0:
            _, _peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()
            peak_mb = _peak / (1024 ** 2)
        times.append(_t1 - _t0)
    median_ms_per_sample = (np.median(times) / _n_test) * 1000
    return round(median_ms_per_sample, 4), round(peak_mb, 2)

# ── Helper: measure training time ────────────────────────────
def _measure_training(train_fn):
    """Returns training time in seconds."""
    _t0 = time.perf_counter()
    train_fn()
    return round(time.perf_counter() - _t0, 2)

# ── Storage: results dict ─────────────────────────────────────
_cost_results = {}

# ─────────────────────────────────────────────────────────────
# BASE LEARNERS
# Re-fit a fresh copy so we measure true training time
# (original models are already checkpointed — we time a re-fit)
# ─────────────────────────────────────────────────────────────
print("  Measuring base learners...")

_base_specs_cost = {
    'CatBoost':     base_learner_specs['CatBoost'],
    'XGBoost':      base_learner_specs['XGBoost'],
    'LightGBM':     base_learner_specs['LightGBM'],
    'Random Forest':base_learner_specs['Random Forest'],
    'Extra Trees':  base_learner_specs['Extra Trees'],
}

for _nm, _spec in _base_specs_cost.items():
    print(f"    {_nm:<18}", end=' ', flush=True)
    _m = _copy2.deepcopy(_spec)

    # Training time
    if _nm == 'XGBoost':
        _train_t = _measure_training(
            lambda m=_m: m.fit(X_train_res, y_train_res, sample_weight=xgb_sw)
        )
    else:
        _train_t = _measure_training(
            lambda m=_m: m.fit(X_train_res, y_train_res)
        )

    # Inference time + memory — use already-trained base_learners
    _fitted = base_learners[_nm]
    _inf_ms, _mem_mb = _measure_inference(
        lambda f=_fitted: f.predict_proba(X_test_sel)
    )

    _cost_results[_nm] = {
        'Train_Time_s':    _train_t,
        'Infer_ms_sample': _inf_ms,
        'Peak_Mem_MB':     _mem_mb,
        'Type':            'Base Learner',
    }
    print(f"train={_train_t:.1f}s | infer={_inf_ms:.4f}ms/sample | mem={_mem_mb:.1f}MB")

# ─────────────────────────────────────────────────────────────
# STANDARD STACKING
# ─────────────────────────────────────────────────────────────
print(f"\n    {'Std. Stacking':<18}", end=' ', flush=True)

# Training: re-fit a copy
_stack_copy = _copy2.deepcopy(stack_model)
_stack_train_t = _measure_training(
    lambda: _stack_copy.fit(X_train_res, y_train_res)
)

_stack_inf_ms, _stack_mem = _measure_inference(
    lambda: stack_model.predict_proba(X_test_sel)
)
_cost_results['Std. Stacking'] = {
    'Train_Time_s':    _stack_train_t,
    'Infer_ms_sample': _stack_inf_ms,
    'Peak_Mem_MB':     _stack_mem,
    'Type':            'Stacking',
}
print(f"train={_stack_train_t:.1f}s | infer={_stack_inf_ms:.4f}ms/sample | mem={_stack_mem:.1f}MB")

# ─────────────────────────────────────────────────────────────
# KNORA-E, KNORA-U, META-DES
# Training: re-fit fresh copies on X_train_unscaled
# ─────────────────────────────────────────────────────────────
print(f"\n    Measuring DES baselines...")

# Use each method's tuned (K, Hc) from CELL 19c/19f, not the old defaults —
# the timing should reflect the configuration actually reported in Table 5.
_des_models_cost = [
    ('KNORA-E',  knora_e,  KNORAE,  K_KNORA_E_OPT, {}),
    ('KNORA-U',  knora_u,  KNORAU,  K_KNORA_U_OPT, {}),
    ('META-DES', meta_des, METADES, K_META_OPT,    {'Hc': HC_META_OPT}),
]

_pool_cost = [
    base_learners['CatBoost'],
    base_learners['XGBoost'],
    base_learners['LightGBM'],
    base_learners['Random Forest'],
    base_learners['Extra Trees'],
]

for _nm, _fitted_des, _DESClass, _k_use, _kwargs in _des_models_cost:
    print(f"    {_nm:<18}", end=' ', flush=True)

    # Training time: re-fit fresh instance
    _des_fresh = _DESClass(
        pool_classifiers=_pool_cost,
        k=_k_use,
        DFP=False, with_IH=False, safe_k=5,
        random_state=42, **_kwargs
    )
    _des_train_t = _measure_training(
        lambda d=_des_fresh: d.fit(X_train_unscaled, y_train_res)
    )

    # Inference: use already-fitted instance
    _des_inf_ms, _des_mem = _measure_inference(
        lambda f=_fitted_des: f.predict_proba(X_test_unscaled)
    )
    _cost_results[_nm] = {
        'Train_Time_s':    _des_train_t,
        'Infer_ms_sample': _des_inf_ms,
        'Peak_Mem_MB':     _des_mem,
        'Type':            'DES',
    }
    print(f"train={_des_train_t:.1f}s | infer={_des_inf_ms:.4f}ms/sample | mem={_des_mem:.1f}MB")


print(f"\n    {'CFL (Proposed)':<18}", end=' ', flush=True)

# Inference time — full cfl_predict_proba pipeline
_cfl_inf_ms, _cfl_mem = _measure_inference(
    lambda: cfl_predict_proba(
        X_test_sel,
        X_test_sel_scaled,
        base_learners,
        shap_test_values,
        mean_shap_profile,
        knn_purity,
        y_train_res,
        n_classes,
        ALPHA, BETA, GAMMA,
        K_NEIGHBORS,
        apply_smooth=True
    )
)

# Training time: we cannot re-run DE in this cell.
# Report component breakdown from known values.
# Base learners already measured above — sum them.
_base_train_total = sum(
    _cost_results[m]['Train_Time_s']
    for m in ['CatBoost','XGBoost','LightGBM','Random Forest','Extra Trees']
)
# DE optimization is ~1.5 hours = ~5400s (known from experiment)
# Report as string to be transparent
_cfl_train_note = f"{_base_train_total:.0f}s (base) + ~5400s (DE) ≈ 1.5 hrs total"

_cost_results['CFL (Proposed)'] = {
    'Train_Time_s':    _base_train_total + 5400,   # approximate total
    'Infer_ms_sample': _cfl_inf_ms,
    'Peak_Mem_MB':     _cfl_mem,
    'Type':            'Proposed',
    'Train_Note':      _cfl_train_note,
}
print(f"train≈1.5hrs | infer={_cfl_inf_ms:.4f}ms/sample | mem={_cfl_mem:.1f}MB")

# ── Print formatted table ─────────────────────────────────────
print("\n" + "─"*80)
print(f"  {'Model':<22} {'Type':<14} {'Train Time':>12} "
      f"{'Infer (ms/s)':>13} {'Mem (MB)':>10}")
print("  " + "─"*76)

_display_order = [
    'CatBoost', 'XGBoost', 'LightGBM',
    'Random Forest', 'Extra Trees',
    'Std. Stacking',
    'KNORA-E', 'KNORA-U', 'META-DES',
    'CFL (Proposed)',
]

for _mn in _display_order:
    if _mn not in _cost_results: continue
    _r  = _cost_results[_mn]
    _mk = " ◄" if 'CFL' in _mn else ""

    # Format training time
    _tt = _r['Train_Time_s']
    if _tt >= 3600:
        _tt_str = f"~{_tt/3600:.1f} hr"
    elif _tt >= 60:
        _tt_str = f"{_tt/60:.1f} min"
    else:
        _tt_str = f"{_tt:.1f} s"

    print(f"  {_mn:<22} {_r['Type']:<14} {_tt_str:>12} "
          f"{_r['Infer_ms_sample']:>13.4f} {_r['Peak_Mem_MB']:>10.1f}{_mk}")

print("  " + "─"*76)
print("  ms/s = milliseconds per sample | Mem = peak memory during inference")
print("  CFL train time includes DE optimization (~5400s) — one-time cost")

# ── Inference time comparison vs CFL ────────────────────────
print(f"\n── Inference Overhead vs Fastest Baseline ──────────────────")
_fastest_inf = min(
    _cost_results[m]['Infer_ms_sample']
    for m in _cost_results if 'CFL' not in m
)
_fastest_nm  = min(
    (m for m in _cost_results if 'CFL' not in m),
    key=lambda m: _cost_results[m]['Infer_ms_sample']
)
_cfl_inf    = _cost_results['CFL (Proposed)']['Infer_ms_sample']
_overhead    = _cfl_inf / _fastest_inf

print(f"  Fastest baseline : {_fastest_nm} ({_fastest_inf:.4f} ms/sample)")
print(f"  CFL            : {_cfl_inf:.4f} ms/sample")
print(f"  Overhead factor  : {_overhead:.1f}×")

if _overhead < 5:
    print(f"  ✅ CFL inference is only {_overhead:.1f}× slower than the fastest")
    print(f"     → Acceptable for clinical batch processing (lab overnight runs)")
else:
    print(f"  ⚠ CFL is {_overhead:.1f}× slower at inference")
    print(f"     → Frame as batch-processing scenario in paper limitations")

# ── Bar chart: inference time comparison ─────────────────────
_fig_c, _ax_c = plt.subplots(figsize=(12, 5))
_infer_vals   = [_cost_results[m]['Infer_ms_sample'] for m in _display_order
                 if m in _cost_results]
_infer_labels = [m for m in _display_order if m in _cost_results]
_infer_colors = ['crimson' if 'CFL' in m else
                 '#2171b5' if m in ['KNORA-E','KNORA-U','META-DES'] else
                 '#969696' for m in _infer_labels]

_bars_c = _ax_c.bar(range(len(_infer_labels)), _infer_vals,
                     color=_infer_colors, alpha=0.85,
                     edgecolor='white', width=0.6)
for _b, _v in zip(_bars_c, _infer_vals):
    _ax_c.text(_b.get_x()+_b.get_width()/2,
               _b.get_height()+0.0001,
               f'{_v:.3f}', ha='center', va='bottom',
               fontsize=8, fontweight='bold')

_ax_c.set_xticks(range(len(_infer_labels)))
_ax_c.set_xticklabels(
    [m.replace(' (Proposed)','★') for m in _infer_labels],
    rotation=20, ha='right', fontsize=9
)
_ax_c.set_ylabel('Inference Time (ms per sample)', fontsize=11)
_ax_c.set_title('Inference Time Comparison — All Models\n'
                '(Median over 10 runs on test set)',
                fontsize=12, fontweight='bold')
_ax_c.grid(True, axis='y', alpha=0.3)

from matplotlib.patches import Patch as _Patch2
_ax_c.legend(handles=[
    _Patch2(color='crimson',  label='CFL (Proposed)'),
    _Patch2(color='#2171b5',  label='DES Methods'),
    _Patch2(color='#969696',  label='Other Baselines'),
], fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR + 'computational_cost_inference.png',
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()

# ── Save CSV ──────────────────────────────────────────────────
_cost_df = pd.DataFrame({
    m: {
        'Type':          _cost_results[m]['Type'],
        'Train_Time_s':  _cost_results[m]['Train_Time_s'],
        'Infer_ms_per_sample': _cost_results[m]['Infer_ms_sample'],
        'Peak_Mem_MB':   _cost_results[m]['Peak_Mem_MB'],
    }
    for m in _display_order if m in _cost_results
}).T

_cost_df.to_csv(RESULTS_DIR + 'computational_cost.csv')
print(f"\n✓ Cost table saved  → {RESULTS_DIR}computational_cost.csv")
print(f"✓ Bar chart saved   → {RESULTS_DIR}computational_cost_inference.png")
print("✓ Cell 26 complete.")

## Cell 27 — Leave-One-Signal-Out Ablation Study


In [ ]:

print("\n" + "="*65)
print("  LEAVE-ONE-SIGNAL-OUT ABLATION — CF-L (3-signal)")
print("="*65)

_full_weights = {'alpha': ALPHA, 'beta': BETA, 'gamma': GAMMA}

def _renormalized_weights(drop_signal):
    """Zero out one signal, rescale the rest to sum to 1 (ratio preserved)."""
    w = dict(_full_weights)
    w[drop_signal] = 0.0
    remaining_sum = sum(v for k, v in w.items() if k != drop_signal)
    for k in w:
        if k != drop_signal:
            w[k] = w[k] / remaining_sum if remaining_sum > EPS else 0.0
    return w['alpha'], w['beta'], w['gamma']

loso_configs = {
    'Full CF-L':         (ALPHA, BETA, GAMMA),
    'No α (confidence)': _renormalized_weights('alpha'),
    'No β (SHAP+MI)':    _renormalized_weights('beta'),
    'No γ (density)':    _renormalized_weights('gamma'),
}

loso_results = {}
for variant, (a, b, g) in loso_configs.items():
    yp_lo, _ = cfl_predict_proba(
        X_test_sel, X_test_sel_scaled,
        base_learners, shap_test_values, mean_shap_profile,
        knn_purity, y_train_res, n_classes,
        a, b, g, K_NEIGHBORS, apply_smooth=True
    )
    ypred_lo = np.argmax(yp_lo, axis=1)
    loso_results[variant] = {
        'alpha':     round(a, 4),
        'beta':      round(b, 4),
        'gamma':     round(g, 4),
        'Accuracy':  accuracy_score(y_test_raw, ypred_lo),
        'F1-Score':  f1_score(y_test_raw, ypred_lo, average='macro', zero_division=0),
        'MCC':       matthews_corrcoef(y_test_raw, ypred_lo),
        'ROC_AUC':   roc_auc_score(y_test_raw, yp_lo, multi_class='ovr',
                        average='macro', labels=all_labels),
        'Log_Loss':  log_loss(y_test_raw, yp_lo, labels=all_labels),
        'ECE':       compute_ece(y_test_raw, yp_lo),
    }

loso_df = pd.DataFrame(loso_results).T
loso_df[['Accuracy','F1-Score','MCC','ROC_AUC','Log_Loss','ECE']] = \
    loso_df[['Accuracy','F1-Score','MCC','ROC_AUC','Log_Loss','ECE']].astype(float).round(4)
print("\nLeave-One-Signal-Out Results:")
print(loso_df.to_string())

# ── Δ vs Full CF-L (positive = worse when signal removed) ───────
_metric_cols  = ['Accuracy','F1-Score','MCC','ROC_AUC','Log_Loss','ECE']
_full_row     = loso_df.loc['Full CF-L', _metric_cols]
delta_df = pd.DataFrame(index=[k for k in loso_configs if k != 'Full CF-L'],
                         columns=_metric_cols, dtype=float)
for variant in delta_df.index:
    for m in _metric_cols:
        # For Log_Loss/ECE, higher = worse; for the rest, lower = worse.
        if m in ('Log_Loss', 'ECE'):
            delta_df.loc[variant, m] = loso_df.loc[variant, m] - _full_row[m]
        else:
            delta_df.loc[variant, m] = _full_row[m] - loso_df.loc[variant, m]
delta_df = delta_df.round(4)
print("\nDegradation vs Full CF-L (positive = performance lost by removing signal):")
print(delta_df.to_string())

# ── Signal importance ranking (by F1-Score drop) ─────────────────
importance_rank = delta_df['F1-Score'].sort_values(ascending=False)
print("\nSignal importance ranking (by macro-F1 drop when removed):")
for i, (variant, drop) in enumerate(importance_rank.items(), 1):
    sig = variant.replace('No ', '').split(' ')[0]
    print(f"  {i}. {sig:<6} ({variant:<20}) → ΔF1 = {drop:+.4f}")

loso_df.to_csv(RESULTS_DIR + 'loso_ablation_study.csv')
delta_df.to_csv(RESULTS_DIR + 'loso_ablation_deltas.csv')

# ── Plot: metric bars + Δ-drop chart ──────────────────────────────
fig, axes = plt.subplots(1, 6, figsize=(30, 5))
loso_colors = ['#e6550d', '#6baed6', '#9ecae1', '#3182bd']
for ax, metric in zip(axes, _metric_cols):
    vals = loso_df[metric].values
    bars = ax.bar(range(len(loso_df)), vals, color=loso_colors, edgecolor='white')
    ax.set_xticks(range(len(loso_df)))
    ax.set_xticklabels(loso_df.index, rotation=30, ha='right', fontsize=8)
    ax.set_title(metric, fontsize=11)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
                f'{val:.4f}', ha='center', va='bottom', fontsize=7)
    ax.grid(True, axis='y', alpha=0.3)
    bars[0].set_edgecolor('crimson'); bars[0].set_linewidth(2.5)
plt.suptitle('Leave-One-Signal-Out Ablation — Marginal Contribution of Each Signal\n'
             '(Full CF-L = red border, leftmost bar)', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'loso_ablation_plot.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

fig2, ax2 = plt.subplots(figsize=(9, 5))
_bar_colors = ['crimson' if v > 0 else '#3182bd' for v in importance_rank.values]
bars2 = ax2.barh(list(importance_rank.index)[::-1], list(importance_rank.values)[::-1],
                  color=_bar_colors[::-1], edgecolor='white')
for bar, val in zip(bars2, list(importance_rank.values)[::-1]):
    ax2.text(bar.get_width(), bar.get_y()+bar.get_height()/2,
              f' {val:+.4f}', va='center', fontsize=9)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_xlabel('ΔMacro-F1 vs Full CF-L (positive = signal is necessary)')
ax2.set_title('Signal Importance Ranking — Leave-One-Signal-Out')
ax2.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'loso_ablation_ranking.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

print(f"\n✓ LOSO ablation table saved  → {RESULTS_DIR}loso_ablation_study.csv")
print(f"✓ LOSO delta table saved     → {RESULTS_DIR}loso_ablation_deltas.csv")
print(f"✓ LOSO plots saved           → {RESULTS_DIR}loso_ablation_plot.png, loso_ablation_ranking.png")
print("✓ Cell 27 complete.")